In [1]:
# Test yfinance is working correctly
import yfinance as yf

ticker = "AAPL"
stock = yf.Ticker(ticker)

# Test 1 — company info
info = stock.info
print("Company Name:", info.get("longName"))
print("Sector:", info.get("sector"))
print("Market Cap:", info.get("marketCap"))

Company Name: Apple Inc.
Sector: Technology
Market Cap: 4532958920704


In [6]:
# Test financial statements
import pandas as pd

# Annual income statement
annual_income = stock.financials
print("ANNUAL INCOME STATEMENT")
print(annual_income)

ANNUAL INCOME STATEMENT
                                                      2025-09-30  \
Tax Effect Of Unusual Items                         0.000000e+00   
Tax Rate For Calcs                                  1.560000e-01   
Normalized EBITDA                                   1.447480e+11   
Net Income From Continuing Operation Net Minori...  1.120100e+11   
Reconciled Depreciation                             1.169800e+10   
Reconciled Cost Of Revenue                          2.209600e+11   
EBITDA                                              1.447480e+11   
EBIT                                                1.330500e+11   
Net Interest Income                                          NaN   
Interest Expense                                             NaN   
Interest Income                                              NaN   
Normalized Income                                   1.120100e+11   
Net Income From Continuing And Discontinued Ope...  1.120100e+11   
Total Expenses          

In [7]:
# Test balance sheet and cash flow
annual_balance = stock.balance_sheet
annual_cashflow = stock.cashflow

print("BALANCE SHEET ROWS:")
print(annual_balance.index.tolist())

print("\nCASH FLOW ROWS:")
print(annual_cashflow.index.tolist())

BALANCE SHEET ROWS:
['Treasury Shares Number', 'Ordinary Shares Number', 'Share Issued', 'Net Debt', 'Total Debt', 'Tangible Book Value', 'Invested Capital', 'Working Capital', 'Net Tangible Assets', 'Capital Lease Obligations', 'Common Stock Equity', 'Total Capitalization', 'Total Equity Gross Minority Interest', 'Stockholders Equity', 'Gains Losses Not Affecting Retained Earnings', 'Other Equity Adjustments', 'Retained Earnings', 'Capital Stock', 'Common Stock', 'Total Liabilities Net Minority Interest', 'Total Non Current Liabilities Net Minority Interest', 'Other Non Current Liabilities', 'Tradeand Other Payables Non Current', 'Long Term Debt And Capital Lease Obligation', 'Long Term Capital Lease Obligation', 'Long Term Debt', 'Current Liabilities', 'Other Current Liabilities', 'Current Deferred Liabilities', 'Current Deferred Revenue', 'Current Debt And Capital Lease Obligation', 'Current Capital Lease Obligation', 'Current Debt', 'Other Current Borrowings', 'Commercial Paper', '

In [8]:
# Extract the specific fields we need and check for NaN
import numpy as np

# Most recent year column (first column)
latest_year = annual_income.columns[0]

# Fields to extract from each statement
fields_income = ['EBIT', 'EBITDA', 'Total Revenue', 'Interest Expense', 'Net Income']
fields_balance = ['Total Debt', 'Net Debt', 'Cash And Cash Equivalents', 
                  'Total Assets', 'Current Assets', 'Current Liabilities', 
                  'Working Capital', 'Retained Earnings']
fields_cashflow = ['Operating Cash Flow', 'Free Cash Flow', 
                   'Capital Expenditure', 'Interest Paid Supplemental Data']

print("=== INCOME STATEMENT (Latest Year) ===")
for field in fields_income:
    try:
        value = annual_income.loc[field, latest_year]
        print(f"{field}: {value:,.0f}" if not pd.isna(value) else f"{field}: NOT AVAILABLE")
    except KeyError:
        print(f"{field}: FIELD MISSING")

print("\n=== BALANCE SHEET (Latest Year) ===")
for field in fields_balance:
    try:
        value = annual_balance.loc[field, latest_year]
        print(f"{field}: {value:,.0f}" if not pd.isna(value) else f"{field}: NOT AVAILABLE")
    except KeyError:
        print(f"{field}: FIELD MISSING")

print("\n=== CASH FLOW (Latest Year) ===")
for field in fields_cashflow:
    try:
        value = annual_cashflow.loc[field, latest_year]
        print(f"{field}: {value:,.0f}" if not pd.isna(value) else f"{field}: NOT AVAILABLE")
    except KeyError:
        print(f"{field}: FIELD MISSING")

=== INCOME STATEMENT (Latest Year) ===
EBIT: 133,050,000,000
EBITDA: 144,748,000,000
Total Revenue: 416,161,000,000
Interest Expense: NOT AVAILABLE
Net Income: 112,010,000,000

=== BALANCE SHEET (Latest Year) ===
Total Debt: 98,657,000,000
Net Debt: 62,723,000,000
Cash And Cash Equivalents: 35,934,000,000
Total Assets: 359,241,000,000
Current Assets: 147,957,000,000
Current Liabilities: 165,631,000,000
Working Capital: -17,674,000,000
Retained Earnings: -14,264,000,000

=== CASH FLOW (Latest Year) ===
Operating Cash Flow: 111,482,000,000
Free Cash Flow: 98,767,000,000
Capital Expenditure: -12,715,000,000
Interest Paid Supplemental Data: NOT AVAILABLE


In [9]:
# Handle Interest Expense fallback + compute all ratios

def get_interest_expense(income_stmt):
    """
    Try to get interest expense from multiple possible fields.
    Falls back across years if latest year is NaN.
    """
    # Try direct Interest Expense first
    for col in income_stmt.columns:
        try:
            val = income_stmt.loc['Interest Expense', col]
            if not pd.isna(val) and val != 0:
                print(f"Interest Expense found in {col}: {val:,.0f}")
                return abs(val)  # make positive
        except KeyError:
            break
    
    # Try Interest Expense Non Operating
    for col in income_stmt.columns:
        try:
            val = income_stmt.loc['Interest Expense Non Operating', col]
            if not pd.isna(val) and val != 0:
                print(f"Interest Expense Non Operating found in {col}: {val:,.0f}")
                return abs(val)
        except KeyError:
            break
    
    print("Interest Expense: NOT FOUND in any year")
    return None

# Get latest year values
latest_year = annual_income.columns[0]

def safe_get(df, field, col):
    """Safely extract a field, returning None if missing or NaN."""
    try:
        val = df.loc[field, col]
        return float(val) if not pd.isna(val) else None
    except KeyError:
        return None

# Extract all fields
ebit        = safe_get(annual_income, 'EBIT', latest_year)
ebitda      = safe_get(annual_income, 'EBITDA', latest_year)
revenue     = safe_get(annual_income, 'Total Revenue', latest_year)
net_income  = safe_get(annual_income, 'Net Income', latest_year)
total_debt  = safe_get(annual_balance, 'Total Debt', latest_year)
net_debt    = safe_get(annual_balance, 'Net Debt', latest_year)
cash        = safe_get(annual_balance, 'Cash And Cash Equivalents', latest_year)
total_assets = safe_get(annual_balance, 'Total Assets', latest_year)
current_assets = safe_get(annual_balance, 'Current Assets', latest_year)
current_liabilities = safe_get(annual_balance, 'Current Liabilities', latest_year)
working_capital = safe_get(annual_balance, 'Working Capital', latest_year)
retained_earnings = safe_get(annual_balance, 'Retained Earnings', latest_year)
operating_cf = safe_get(annual_cashflow, 'Operating Cash Flow', latest_year)
free_cf      = safe_get(annual_cashflow, 'Free Cash Flow', latest_year)
capex        = safe_get(annual_cashflow, 'Capital Expenditure', latest_year)
interest_expense = get_interest_expense(annual_income)

# Now compute ratios
print("\n=== COMPUTED RATIOS ===")

# 1. Debt / EBITDA
if total_debt and ebitda:
    debt_to_ebitda = total_debt / ebitda
    print(f"Debt/EBITDA: {debt_to_ebitda:.2f}x")
else:
    debt_to_ebitda = None
    print("Debt/EBITDA: NOT AVAILABLE")

# 2. Interest Coverage (EBIT / Interest Expense)
if ebit and interest_expense:
    interest_coverage = ebit / interest_expense
    print(f"Interest Coverage: {interest_coverage:.2f}x")
else:
    interest_coverage = None
    print("Interest Coverage: NOT AVAILABLE — Interest Expense missing")

# 3. Current Ratio
if current_assets and current_liabilities:
    current_ratio = current_assets / current_liabilities
    print(f"Current Ratio: {current_ratio:.2f}x")
else:
    current_ratio = None
    print("Current Ratio: NOT AVAILABLE")

# 4. Free Cash Flow to Debt
if free_cf and total_debt:
    fcf_to_debt = free_cf / total_debt
    print(f"FCF to Debt: {fcf_to_debt:.2f}x")
else:
    fcf_to_debt = None
    print("FCF to Debt: NOT AVAILABLE")

# 5. Net Debt / EBITDA
if net_debt and ebitda:
    net_debt_to_ebitda = net_debt / ebitda
    print(f"Net Debt/EBITDA: {net_debt_to_ebitda:.2f}x")
else:
    net_debt_to_ebitda = None
    print("Net Debt/EBITDA: NOT AVAILABLE")

# 6. DSCR (Operating CF / Total Debt)
# Using Operating CF / Total Debt as proxy since current debt service not always available
if operating_cf and total_debt:
    dscr = operating_cf / total_debt
    print(f"DSCR (proxy): {dscr:.2f}x")
else:
    dscr = None
    print("DSCR: NOT AVAILABLE")

# 7. Altman Z-Score
# Z = 1.2*(WC/TA) + 1.4*(RE/TA) + 3.3*(EBIT/TA) + 0.6*(MktCap/TotalDebt) + 1.0*(Rev/TA)
market_cap = stock.info.get('marketCap')
if all([working_capital, retained_earnings, ebit, market_cap, total_debt, revenue, total_assets]):
    z1 = 1.2 * (working_capital / total_assets)
    z2 = 1.4 * (retained_earnings / total_assets)
    z3 = 3.3 * (ebit / total_assets)
    z4 = 0.6 * (market_cap / total_debt)
    z5 = 1.0 * (revenue / total_assets)
    altman_z = z1 + z2 + z3 + z4 + z5
    if altman_z > 2.99:
        zone = "SAFE"
    elif altman_z > 1.81:
        zone = "GREY"
    else:
        zone = "DISTRESS"
    print(f"Altman Z-Score: {altman_z:.2f} ({zone})")
else:
    altman_z = None
    print("Altman Z-Score: NOT AVAILABLE")

Interest Expense found in 2023-09-30 00:00:00: 3,933,000,000

=== COMPUTED RATIOS ===
Debt/EBITDA: 0.68x
Interest Coverage: 33.83x
Current Ratio: 0.89x
FCF to Debt: 1.00x
Net Debt/EBITDA: 0.43x
DSCR (proxy): 1.13x
Altman Z-Score: 29.83 (SAFE)


In [10]:
def get_financial_ratios(ticker: str) -> dict:
    """
    Pulls financial statements from yfinance and computes credit ratios.
    
    Input: ticker (str) — e.g. "AAPL"
    Output: dict with keys:
        - company_info: dict (name, sector, market cap)
        - ratios: dict (all seven ratios, None if unavailable)
        - raw: dict (raw financial values used in calculations)
        - latest_year: str (fiscal year of latest data)
        - status: dict (flags for what data was available)
    """
    
    result = {
        "company_info": {},
        "ratios": {},
        "raw": {},
        "latest_year": None,
        "status": {}
    }
    
    try:
        stock = yf.Ticker(ticker)
        
        # --- Company Info ---
        try:
            info = stock.info
            result["company_info"] = {
                "name": info.get("longName", "Unknown"),
                "sector": info.get("sector", "Unknown"),
                "industry": info.get("industry", "Unknown"),
                "market_cap": info.get("marketCap", None),
                "description": info.get("longBusinessSummary", "Not available")
            }
        except Exception as e:
            result["status"]["company_info_error"] = str(e)
            result["company_info"] = {
                "name": ticker,
                "sector": "Unknown",
                "industry": "Unknown", 
                "market_cap": None,
                "description": "Not available"
            }

        # --- Financial Statements ---
        try:
            annual_income  = stock.financials
            annual_balance = stock.balance_sheet
            annual_cashflow = stock.cashflow
            latest_year = annual_income.columns[0]
            result["latest_year"] = str(latest_year.date())
        except Exception as e:
            result["status"]["financials_error"] = str(e)
            return result

        # --- Safe Extraction ---
        def safe_get(df, field, col):
            try:
                val = df.loc[field, col]
                return float(val) if not pd.isna(val) else None
            except KeyError:
                return None

        # --- Interest Expense Fallback ---
        def get_interest_expense(income_stmt):
            for col in income_stmt.columns:
                for field in ['Interest Expense', 'Interest Expense Non Operating']:
                    try:
                        val = income_stmt.loc[field, col]
                        if val is not None and not pd.isna(val) and val != 0:
                            return abs(float(val))
                    except KeyError:
                        continue
            return None

        # --- Extract Raw Values ---
        ebit               = safe_get(annual_income, 'EBIT', latest_year)
        ebitda             = safe_get(annual_income, 'EBITDA', latest_year)
        revenue            = safe_get(annual_income, 'Total Revenue', latest_year)
        net_income         = safe_get(annual_income, 'Net Income', latest_year)
        total_debt         = safe_get(annual_balance, 'Total Debt', latest_year)
        net_debt           = safe_get(annual_balance, 'Net Debt', latest_year)
        cash               = safe_get(annual_balance, 'Cash And Cash Equivalents', latest_year)
        total_assets       = safe_get(annual_balance, 'Total Assets', latest_year)
        current_assets     = safe_get(annual_balance, 'Current Assets', latest_year)
        current_liabilities = safe_get(annual_balance, 'Current Liabilities', latest_year)
        working_capital    = safe_get(annual_balance, 'Working Capital', latest_year)
        retained_earnings  = safe_get(annual_balance, 'Retained Earnings', latest_year)
        operating_cf       = safe_get(annual_cashflow, 'Operating Cash Flow', latest_year)
        free_cf            = safe_get(annual_cashflow, 'Free Cash Flow', latest_year)
        interest_expense   = get_interest_expense(annual_income)
        market_cap         = result["company_info"]["market_cap"]

        result["raw"] = {
            "ebit": ebit,
            "ebitda": ebitda,
            "revenue": revenue,
            "net_income": net_income,
            "total_debt": total_debt,
            "net_debt": net_debt,
            "cash": cash,
            "total_assets": total_assets,
            "current_assets": current_assets,
            "current_liabilities": current_liabilities,
            "working_capital": working_capital,
            "retained_earnings": retained_earnings,
            "operating_cf": operating_cf,
            "free_cf": free_cf,
            "interest_expense": interest_expense,
            "market_cap": market_cap
        }

        # --- Compute Ratios ---
        ratios = {}

        # 1. Debt / EBITDA
        ratios["debt_to_ebitda"] = (
            round(total_debt / ebitda, 2) 
            if total_debt and ebitda 
            else None
        )

        # 2. Interest Coverage
        ratios["interest_coverage"] = (
            round(ebit / interest_expense, 2) 
            if ebit and interest_expense 
            else None
        )

        # 3. Current Ratio
        ratios["current_ratio"] = (
            round(current_assets / current_liabilities, 2) 
            if current_assets and current_liabilities 
            else None
        )

        # 4. FCF to Debt
        ratios["fcf_to_debt"] = (
            round(free_cf / total_debt, 2) 
            if free_cf and total_debt 
            else None
        )

        # 5. Net Debt / EBITDA
        ratios["net_debt_to_ebitda"] = (
            round(net_debt / ebitda, 2) 
            if net_debt and ebitda 
            else None
        )

        # 6. DSCR
        ratios["dscr"] = (
            round(operating_cf / total_debt, 2) 
            if operating_cf and total_debt 
            else None
        )

        # 7. Altman Z-Score
        if all([
            working_capital, retained_earnings, ebit,
            market_cap, total_debt, revenue, total_assets
        ]):
            z = (
                1.2 * (working_capital / total_assets) +
                1.4 * (retained_earnings / total_assets) +
                3.3 * (ebit / total_assets) +
                0.6 * (market_cap / total_debt) +
                1.0 * (revenue / total_assets)
            )
            ratios["altman_z"] = round(z, 2)
            ratios["altman_zone"] = (
                "SAFE" if z > 2.99 else 
                "GREY" if z > 1.81 else 
                "DISTRESS"
            )
        else:
            ratios["altman_z"] = None
            ratios["altman_zone"] = None

        result["ratios"] = ratios
        result["status"]["success"] = True

    except Exception as e:
        result["status"]["error"] = str(e)
        result["status"]["success"] = False

    return result


# --- Test it ---
data = get_financial_ratios("AAPL")

print(f"Company: {data['company_info']['name']}")
print(f"Sector: {data['company_info']['sector']}")
print(f"Latest Year: {data['latest_year']}")
print(f"\nRATIOS:")
for k, v in data['ratios'].items():
    print(f"  {k}: {v}")
print(f"\nStatus: {data['status']}")

Company: Apple Inc.
Sector: Technology
Latest Year: 2025-09-30

RATIOS:
  debt_to_ebitda: 0.68
  interest_coverage: 33.83
  current_ratio: 0.89
  fcf_to_debt: 1.0
  net_debt_to_ebitda: 0.43
  dscr: 1.13
  altman_z: 29.83
  altman_zone: SAFE

Status: {'success': True}


In [11]:
# Stress test on two more tickers
# Boeing (BA) — stressed industrial company with real debt concerns
# JPMorgan (JPM) — large cap financial, will test how function handles banks

test_tickers = ["BA", "JPM"]

for ticker in test_tickers:
    print(f"\n{'='*50}")
    print(f"Testing: {ticker}")
    print('='*50)
    
    data = get_financial_ratios(ticker)
    
    print(f"Company: {data['company_info']['name']}")
    print(f"Sector: {data['company_info']['sector']}")
    print(f"Latest Year: {data['latest_year']}")
    print(f"\nRATIOS:")
    for k, v in data['ratios'].items():
        print(f"  {k}: {v}")
    print(f"Status: {data['status']}")
    


Testing: BA
Company: The Boeing Company
Sector: Industrials
Latest Year: 2025-12-31

RATIOS:
  debt_to_ebitda: 7.4
  interest_coverage: 1.95
  current_ratio: 1.19
  fcf_to_debt: -0.03
  net_debt_to_ebitda: 5.83
  dscr: 0.02
  altman_z: 2.89
  altman_zone: GREY
Status: {'success': True}

Testing: JPM
Company: JPMorgan Chase & Co.
Sector: Financial Services
Latest Year: 2025-12-31

RATIOS:
  debt_to_ebitda: None
  interest_coverage: None
  current_ratio: None
  fcf_to_debt: -0.3
  net_debt_to_ebitda: None
  dscr: -0.3
  altman_z: None
  altman_zone: None
Status: {'success': True}


In [12]:
# Add sector exclusion to the function
# We add this check right at the start, before any ratio computation

def get_financial_ratios(ticker: str) -> dict:
    """
    Pulls financial statements from yfinance and computes credit ratios.
    
    Input: ticker (str) — e.g. "AAPL"
    Output: dict with keys:
        - company_info: dict (name, sector, market cap)
        - ratios: dict (all seven ratios, None if unavailable)
        - raw: dict (raw financial values used in calculations)
        - latest_year: str (fiscal year of latest data)
        - status: dict (flags for what data was available)
    """
    
    # Sectors excluded from analysis
    EXCLUDED_SECTORS = [
        "Financial Services",
        "Insurance",
        "Banks",
        "Diversified Financials"
    ]

    result = {
        "company_info": {},
        "ratios": {},
        "raw": {},
        "latest_year": None,
        "status": {}
    }

    try:
        stock = yf.Ticker(ticker)

        # --- Company Info First (needed for sector check) ---
        try:
            info = stock.info
            sector = info.get("sector", "Unknown")
            result["company_info"] = {
                "name": info.get("longName", "Unknown"),
                "sector": sector,
                "industry": info.get("industry", "Unknown"),
                "market_cap": info.get("marketCap", None),
                "description": info.get("longBusinessSummary", "Not available")
            }
        except Exception as e:
            result["status"]["company_info_error"] = str(e)
            result["status"]["success"] = False
            return result

        # --- Sector Check ---
        if sector in EXCLUDED_SECTORS:
            result["status"]["success"] = False
            result["status"]["error"] = (
                f"Sector '{sector}' is not supported. "
                f"This tool is designed for non-financial corporates only. "
                f"Financial institutions use different credit frameworks "
                f"(Tier 1 Capital, NIM, NPL ratio) outside this system's scope."
            )
            return result

        # --- Market Cap Check (large cap only) ---
        market_cap = result["company_info"]["market_cap"]
        if market_cap and market_cap < 10_000_000_000:  # below $10bn
            result["status"]["success"] = False
            result["status"]["error"] = (
                f"Market cap ${market_cap/1e9:.1f}bn is below the $10bn "
                f"large cap threshold. This tool is designed for large cap "
                f"companies only."
            )
            return result

        # --- Financial Statements ---
        try:
            annual_income   = stock.financials
            annual_balance  = stock.balance_sheet
            annual_cashflow = stock.cashflow
            latest_year     = annual_income.columns[0]
            result["latest_year"] = str(latest_year.date())
        except Exception as e:
            result["status"]["financials_error"] = str(e)
            result["status"]["success"] = False
            return result

        # --- Safe Extraction ---
        def safe_get(df, field, col):
            try:
                val = df.loc[field, col]
                return float(val) if not pd.isna(val) else None
            except KeyError:
                return None

        # --- Interest Expense Fallback ---
        def get_interest_expense(income_stmt):
            for col in income_stmt.columns:
                for field in ['Interest Expense', 'Interest Expense Non Operating']:
                    try:
                        val = income_stmt.loc[field, col]
                        if val is not None and not pd.isna(val) and val != 0:
                            return abs(float(val))
                    except KeyError:
                        continue
            return None

        # --- Extract Raw Values ---
        ebit                = safe_get(annual_income, 'EBIT', latest_year)
        ebitda              = safe_get(annual_income, 'EBITDA', latest_year)
        revenue             = safe_get(annual_income, 'Total Revenue', latest_year)
        net_income          = safe_get(annual_income, 'Net Income', latest_year)
        total_debt          = safe_get(annual_balance, 'Total Debt', latest_year)
        net_debt            = safe_get(annual_balance, 'Net Debt', latest_year)
        cash                = safe_get(annual_balance, 'Cash And Cash Equivalents', latest_year)
        total_assets        = safe_get(annual_balance, 'Total Assets', latest_year)
        current_assets      = safe_get(annual_balance, 'Current Assets', latest_year)
        current_liabilities = safe_get(annual_balance, 'Current Liabilities', latest_year)
        working_capital     = safe_get(annual_balance, 'Working Capital', latest_year)
        retained_earnings   = safe_get(annual_balance, 'Retained Earnings', latest_year)
        operating_cf        = safe_get(annual_cashflow, 'Operating Cash Flow', latest_year)
        free_cf             = safe_get(annual_cashflow, 'Free Cash Flow', latest_year)
        interest_expense    = get_interest_expense(annual_income)
        market_cap          = result["company_info"]["market_cap"]

        result["raw"] = {
            "ebit": ebit,
            "ebitda": ebitda,
            "revenue": revenue,
            "net_income": net_income,
            "total_debt": total_debt,
            "net_debt": net_debt,
            "cash": cash,
            "total_assets": total_assets,
            "current_assets": current_assets,
            "current_liabilities": current_liabilities,
            "working_capital": working_capital,
            "retained_earnings": retained_earnings,
            "operating_cf": operating_cf,
            "free_cf": free_cf,
            "interest_expense": interest_expense,
            "market_cap": market_cap
        }

        # --- Compute Ratios ---
        ratios = {}

        ratios["debt_to_ebitda"] = (
            round(total_debt / ebitda, 2)
            if total_debt and ebitda
            else None
        )

        ratios["interest_coverage"] = (
            round(ebit / interest_expense, 2)
            if ebit and interest_expense
            else None
        )

        ratios["current_ratio"] = (
            round(current_assets / current_liabilities, 2)
            if current_assets and current_liabilities
            else None
        )

        ratios["fcf_to_debt"] = (
            round(free_cf / total_debt, 2)
            if free_cf and total_debt
            else None
        )

        ratios["net_debt_to_ebitda"] = (
            round(net_debt / ebitda, 2)
            if net_debt and ebitda
            else None
        )

        ratios["dscr"] = (
            round(operating_cf / total_debt, 2)
            if operating_cf and total_debt
            else None
        )

        if all([
            working_capital, retained_earnings, ebit,
            market_cap, total_debt, revenue, total_assets
        ]):
            z = (
                1.2 * (working_capital / total_assets) +
                1.4 * (retained_earnings / total_assets) +
                3.3 * (ebit / total_assets) +
                0.6 * (market_cap / total_debt) +
                1.0 * (revenue / total_assets)
            )
            ratios["altman_z"]    = round(z, 2)
            ratios["altman_zone"] = (
                "SAFE"     if z > 2.99 else
                "GREY"     if z > 1.81 else
                "DISTRESS"
            )
        else:
            ratios["altman_z"]    = None
            ratios["altman_zone"] = None

        result["ratios"] = ratios
        result["status"]["success"] = True

    except Exception as e:
        result["status"]["error"] = str(e)
        result["status"]["success"] = False

    return result


# --- Test all three cases ---
test_cases = {
    "AAPL": "Should work — large cap tech",
    "JPM":  "Should be blocked — financial sector",
    "BA":   "Should work — stressed industrial",
}

for ticker, description in test_cases.items():
    print(f"\n{'='*50}")
    print(f"{ticker} — {description}")
    print('='*50)
    data = get_financial_ratios(ticker)
    if not data["status"].get("success"):
        print(f"BLOCKED: {data['status'].get('error')}")
    else:
        print(f"Company: {data['company_info']['name']}")
        print(f"Latest Year: {data['latest_year']}")
        for k, v in data["ratios"].items():
            print(f"  {k}: {v}")


AAPL — Should work — large cap tech
Company: Apple Inc.
Latest Year: 2025-09-30
  debt_to_ebitda: 0.68
  interest_coverage: 33.83
  current_ratio: 0.89
  fcf_to_debt: 1.0
  net_debt_to_ebitda: 0.43
  dscr: 1.13
  altman_z: 29.83
  altman_zone: SAFE

JPM — Should be blocked — financial sector
BLOCKED: Sector 'Financial Services' is not supported. This tool is designed for non-financial corporates only. Financial institutions use different credit frameworks (Tier 1 Capital, NIM, NPL ratio) outside this system's scope.

BA — Should work — stressed industrial
Company: The Boeing Company
Latest Year: 2025-12-31
  debt_to_ebitda: 7.4
  interest_coverage: 1.95
  current_ratio: 1.19
  fcf_to_debt: -0.03
  net_debt_to_ebitda: 5.83
  dscr: 0.02
  altman_z: 2.89
  altman_zone: GREY


In [13]:
# Verify the .py file works by importing from it
import importlib
import sys
sys.path.append('..')  # so notebook can find src/

import src.data.financials as fin
importlib.reload(fin)

data = fin.get_financial_ratios("AAPL")
print("Import test:", "PASSED" if data["status"]["success"] else "FAILED")
print("Company:", data["company_info"]["name"])
print("Ratios:", data["ratios"])

Import test: PASSED
Company: Apple Inc.
Ratios: {'debt_to_ebitda': 0.68, 'interest_coverage': 33.83, 'current_ratio': 0.89, 'fcf_to_debt': 1.0, 'net_debt_to_ebitda': 0.43, 'dscr': 1.13, 'altman_z': 29.83, 'altman_zone': 'SAFE'}


In [ ]:
# Test EDGAR company search
# We need to find the CIK number for a given company name
# CIK is EDGAR's internal identifier — different from ticker

import requests

def search_edgar_cik(company_name: str) -> str:
    """
    Search EDGAR for a company's CIK number using their legal name.
    Returns CIK as a string, or None if not found.
    """
    url = "https://efts.sec.gov/LATEST/search-index?q={}&dateRange=custom&startdt=2020-01-01&enddt=2024-01-01&forms=10-K".format(company_name)
    
    headers = {
        "User-Agent": "credit-committee-project YOUR_ACTUAL_EMAIL@gmail.com"
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        print("Status code:", response.status_code)
        print("Response preview:", response.text[:500])
    except Exception as e:
        print("Error:", e)

# Test with Apple
search_edgar_cik("Apple Inc")

Status code: 200
Response preview: {"took":953,"timed_out":false,"_shards":{"total":50,"successful":50,"skipped":0,"failed":0},"hits":{"total":{"value":2552,"relation":"eq"},"max_score":9.510699,"hits":[{"_index":"edgar_file","_id":"0000950170-23-003449:aple-ex21_1.htm","_score":9.510699,"_source":{"ciks":["0001418121"],"period_ending":"2022-12-31","file_num":["001-37389"],"display_names":["Apple Hospitality REIT, Inc.  (APLE)  (CIK 0001418121)"],"xsl":null,"sequence":3,"root_forms":["10-K"],"file_date":"2023-02-21","biz_states":


In [ ]:
import requests

def search_edgar_cik(ticker: str) -> str:
    """
    Search EDGAR for a company's CIK number using ticker.
    Returns CIK as a string, or None if not found.
    """
    headers = {
        "User-Agent": "credit-committee-project YOUR_ACTUAL_EMAIL@gmail.com"
    }
    
    try:
        # EDGAR's company ticker to CIK mapping — most reliable method
        response = requests.get(
            "https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={}&type=10-K&dateb=&owner=include&count=5&search_text=&output=atom".format(ticker),
            headers=headers,
            timeout=10
        )
        print("Status code:", response.status_code)
        print("Response preview:", response.text[:1000])
    except Exception as e:
        print("Error:", e)

# Test with Apple ticker directly
search_edgar_cik("AAPL")

Status code: 200
Response preview: <?xml version="1.0" encoding="ISO-8859-1" ?>
  <feed xmlns="http://www.w3.org/2005/Atom">
    <author>
      <email>webmaster@sec.gov</email>
      <name>Webmaster</name>
    </author>
    <company-info>
      <addresses>
        <address type="mailing">
          <city>CUPERTINO</city>
          <state>CA</state>
          <street1>ONE APPLE PARK WAY</street1>
          <zip>95014</zip>
        </address>
        <address type="business">
          <city>CUPERTINO</city>
          <phone>(408) 996-1010</phone>
          <state>CA</state>
          <street1>ONE APPLE PARK WAY</street1>
          <zip>95014</zip>
        </address>
      </addresses>
      <assigned-sic>3571</assigned-sic>
      <assigned-sic-desc>ELECTRONIC COMPUTERS</assigned-sic-desc>
      <assigned-sic-href>https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&amp;SIC=3571&amp;owner=include&amp;count=10</assigned-sic-href>
      <cik>0000320193</cik>
      <cik-href>https://w

In [ ]:
import requests
import xml.etree.ElementTree as ET

def get_edgar_cik(ticker: str, user_email: str) -> dict:
    """
    Gets CIK number from EDGAR using ticker symbol.
    
    Input: ticker (str) — e.g. "AAPL"
    Output: dict with keys:
        - cik: str (e.g. "0000320193")
        - company_name: str (legal name from EDGAR)
        - sic: str (industry code)
        - sic_desc: str (industry description)
        - status: dict
    """
    
    result = {
        "cik": None,
        "company_name": None,
        "sic": None,
        "sic_desc": None,
        "status": {}
    }
    
    headers = {
        "User-Agent": f"credit-committee-project {user_email}"
    }
    
    try:
        response = requests.get(
            "https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={}&type=10-K&dateb=&owner=include&count=5&search_text=&output=atom".format(ticker),
            headers=headers,
            timeout=10
        )
        
        if response.status_code != 200:
            result["status"]["success"] = False
            result["status"]["error"] = f"EDGAR returned status code {response.status_code}"
            return result
        
        # Parse XML response
        # Remove default namespace to simplify parsing
        xml_text = response.text.replace(
            'xmlns="http://www.w3.org/2005/Atom"', ''
        )
        root = ET.fromstring(xml_text)
        
        company_info = root.find("company-info")
        
        if company_info is None:
            result["status"]["success"] = False
            result["status"]["error"] = "Company not found on EDGAR"
            return result
        
        result["cik"]          = company_info.find("cik").text
        result["company_name"] = company_info.find("conformed-name").text
        result["sic"]          = company_info.find("assigned-sic").text
        result["sic_desc"]     = company_info.find("assigned-sic-desc").text
        result["status"]["success"] = True
        
    except Exception as e:
        result["status"]["success"] = False
        result["status"]["error"] = str(e)
    
    return result


# Test on three tickers
import os
USER_EMAIL = "YOUR_ACTUAL_EMAIL@gmail.com"  # replace with yours

for ticker in ["AAPL", "BA", "JPM"]:
    print(f"\n{'='*40}")
    print(f"Testing: {ticker}")
    data = get_edgar_cik(ticker, USER_EMAIL)
    if data["status"]["success"]:
        print(f"CIK: {data['cik']}")
        print(f"Legal Name: {data['company_name']}")
        print(f"SIC: {data['sic']} — {data['sic_desc']}")
    else:
        print(f"FAILED: {data['status']['error']}")


Testing: AAPL
CIK: 0000320193
Legal Name: Apple Inc.
SIC: 3571 — ELECTRONIC COMPUTERS

Testing: BA
CIK: 0000012927
Legal Name: BOEING CO
SIC: 3721 — AIRCRAFT

Testing: JPM
CIK: 0000019617
Legal Name: JPMORGAN CHASE & CO
SIC: 6021 — NATIONAL COMMERCIAL BANKS


In [ ]:
import yfinance as yf
import requests
import xml.etree.ElementTree as ET

USER_EMAIL = "YOUR_ACTUAL_EMAIL@gmail.com"  # replace with yours

EXCLUDED_SECTORS = [
    "Financial Services",
    "Insurance", 
    "Banks",
    "Diversified Financials"
]

# SIC codes for financial institutions — backup check
EXCLUDED_SIC_PREFIXES = ["60", "61", "62", "63", "64", "65"]

def resolve_ticker(ticker: str) -> dict:
    """
    Resolves a ticker to all identifiers needed by the pipeline.
    Validates: exists on yfinance, large cap, non-financial sector.
    
    Input: ticker (str) — user provided e.g. "AAPL"
    Output: dict with keys:
        - ticker: str
        - legal_name: str
        - cik: str
        - sector: str
        - industry: str
        - market_cap: float
        - sic: str
        - sic_desc: str
        - status: dict (success bool + error message if failed)
    """
    
    result = {
        "ticker": ticker.upper(),
        "legal_name": None,
        "cik": None,
        "sector": None,
        "industry": None,
        "market_cap": None,
        "sic": None,
        "sic_desc": None,
        "status": {}
    }
    
    # --- Step 1: Validate ticker exists on yfinance ---
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        
        if not info or "longName" not in info:
            result["status"]["success"] = False
            result["status"]["error"] = f"Ticker '{ticker}' not found on Yahoo Finance."
            return result
        
        result["legal_name"] = info.get("longName", "Unknown")
        result["sector"]     = info.get("sector", "Unknown")
        result["industry"]   = info.get("industry", "Unknown")
        result["market_cap"] = info.get("marketCap", None)
        
    except Exception as e:
        result["status"]["success"] = False
        result["status"]["error"] = f"yfinance error: {str(e)}"
        return result
    
    # --- Step 2: Sector check ---
    if result["sector"] in EXCLUDED_SECTORS:
        result["status"]["success"] = False
        result["status"]["error"] = (
            f"Sector '{result['sector']}' is not supported. "
            f"This tool is designed for non-financial corporates only."
        )
        return result
    
    # --- Step 3: Market cap check ---
    if result["market_cap"] and result["market_cap"] < 10_000_000_000:
        result["status"]["success"] = False
        result["status"]["error"] = (
            f"Market cap ${result['market_cap']/1e9:.1f}bn is below "
            f"the $10bn large cap threshold."
        )
        return result
    
    # --- Step 4: Get CIK from EDGAR ---
    try:
        headers = {"User-Agent": f"credit-committee-project {USER_EMAIL}"}
        response = requests.get(
            "https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={}&type=10-K&dateb=&owner=include&count=5&search_text=&output=atom".format(ticker),
            headers=headers,
            timeout=10
        )
        
        if response.status_code == 200:
            xml_text = response.text.replace(
                'xmlns="http://www.w3.org/2005/Atom"', ''
            )
            root = ET.fromstring(xml_text)
            company_info = root.find("company-info")
            
            if company_info is not None:
                result["cik"]      = company_info.find("cik").text
                result["sic"]      = company_info.find("assigned-sic").text
                result["sic_desc"] = company_info.find("assigned-sic-desc").text
                
                # Backup SIC-based financial sector check
                if result["sic"] and any(
                    result["sic"].startswith(prefix) 
                    for prefix in EXCLUDED_SIC_PREFIXES
                ):
                    result["status"]["success"] = False
                    result["status"]["error"] = (
                        f"SIC code {result['sic']} ({result['sic_desc']}) "
                        f"indicates a financial institution. Not supported."
                    )
                    return result
            else:
                # CIK not found but don't block — EDGAR may just not have it
                result["status"]["cik_warning"] = (
                    "CIK not found on EDGAR — filing download may fail."
                )
        else:
            result["status"]["cik_warning"] = (
                f"EDGAR returned {response.status_code} — CIK unavailable."
            )
            
    except Exception as e:
        # EDGAR failure is non-blocking — warn but continue
        result["status"]["cik_warning"] = f"EDGAR lookup failed: {str(e)}"
    
    result["status"]["success"] = True
    return result


# --- Test all cases ---
test_cases = {
    "AAPL": "Valid large cap tech",
    "BA":   "Valid stressed industrial", 
    "JPM":  "Should fail — financial sector",
    "NVDA": "Valid large cap semiconductor",
    "XYZ":  "Should fail — invalid ticker",
}

for ticker, description in test_cases.items():
    print(f"\n{'='*50}")
    print(f"{ticker} — {description}")
    print('='*50)
    data = resolve_ticker(ticker)
    if data["status"]["success"]:
        print(f"Legal Name : {data['legal_name']}")
        print(f"CIK        : {data['cik']}")
        print(f"Sector     : {data['sector']}")
        print(f"Market Cap : ${data['market_cap']/1e9:.1f}bn")
        print(f"SIC        : {data['sic']} — {data['sic_desc']}")
        if "cik_warning" in data["status"]:
            print(f"WARNING    : {data['status']['cik_warning']}")
    else:
        print(f"BLOCKED: {data['status']['error']}")


AAPL — Valid large cap tech
Legal Name : Apple Inc.
CIK        : 0000320193
Sector     : Technology
Market Cap : $4533.0bn
SIC        : 3571 — ELECTRONIC COMPUTERS

BA — Valid stressed industrial
Legal Name : The Boeing Company
CIK        : 0000012927
Sector     : Industrials
Market Cap : $178.5bn
SIC        : 3721 — AIRCRAFT

JPM — Should fail — financial sector
BLOCKED: Sector 'Financial Services' is not supported. This tool is designed for non-financial corporates only.

NVDA — Valid large cap semiconductor
Legal Name : NVIDIA Corporation
CIK        : 0001045810
Sector     : Technology
Market Cap : $4719.0bn
SIC        : 3674 — SEMICONDUCTORS &amp; RELATED DEVICES

XYZ — Should fail — invalid ticker
Legal Name : Block, Inc.
CIK        : None
Sector     : Technology
Market Cap : $46.9bn
SIC        : None — None


In [18]:
# Test with a truly invalid ticker
test_invalid = ["ZZZZ123", "FAKECO"]

for ticker in test_invalid:
    print(f"\nTesting invalid ticker: {ticker}")
    data = resolve_ticker(ticker)
    if data["status"]["success"]:
        print(f"Passed (unexpected): {data['legal_name']}")
    else:
        print(f"BLOCKED: {data['status']['error']}")


Testing invalid ticker: ZZZZ123


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ZZZZ123"}}}


BLOCKED: Ticker 'ZZZZ123' not found on Yahoo Finance.

Testing invalid ticker: FAKECO


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: FAKECO"}}}


BLOCKED: Ticker 'FAKECO' not found on Yahoo Finance.


In [19]:
# Test sec-edgar-downloader
from sec_edgar_downloader import Downloader
import os

# Initialize downloader
# First argument is company name, second is email
dl = Downloader("credit-committee", "YOUR_ACTUAL_EMAIL@gmail.com", "data/filings")

# Download most recent 10-K for Apple
dl.get("10-K", "AAPL", limit=1)

# Check if file was downloaded
for root, dirs, files in os.walk("data/filings"):
    for file in files:
        print(os.path.join(root, file))

data/filings\sec-edgar-filings\AAPL\10-K\0000320193-25-000079\full-submission.txt


In [20]:
# Check file size and preview contents
file_path = "data/filings/sec-edgar-filings/AAPL/10-K/0000320193-25-000079/full-submission.txt"

# File size
size_mb = os.path.getsize(file_path) / (1024 * 1024)
print(f"File size: {size_mb:.1f} MB")

# Preview first 2000 characters
with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    content = f.read()
    print(f"\nTotal characters: {len(content):,}")
    print(f"\nFirst 2000 characters:")
    print(content[:2000])

File size: 9.0 MB

Total characters: 9,392,337

First 2000 characters:
<SEC-DOCUMENT>0000320193-25-000079.txt : 20251031
<SEC-HEADER>0000320193-25-000079.hdr.sgml : 20251031
<ACCEPTANCE-DATETIME>20251031060126
ACCESSION NUMBER:		0000320193-25-000079
CONFORMED SUBMISSION TYPE:	10-K
PUBLIC DOCUMENT COUNT:		91
CONFORMED PERIOD OF REPORT:	20250927
FILED AS OF DATE:		20251031
DATE AS OF CHANGE:		20251031

FILER:

	COMPANY DATA:	
		COMPANY CONFORMED NAME:			Apple Inc.
		CENTRAL INDEX KEY:			0000320193
		STANDARD INDUSTRIAL CLASSIFICATION:	ELECTRONIC COMPUTERS [3571]
		ORGANIZATION NAME:           	06 Technology
		EIN:				942404110
		STATE OF INCORPORATION:			CA
		FISCAL YEAR END:			0927

	FILING VALUES:
		FORM TYPE:		10-K
		SEC ACT:		1934 Act
		SEC FILE NUMBER:	001-36743
		FILM NUMBER:		251437791

	BUSINESS ADDRESS:	
		STREET 1:		ONE APPLE PARK WAY
		CITY:			CUPERTINO
		STATE:			CA
		ZIP:			95014
		BUSINESS PHONE:		(408) 996-1010

	MAIL ADDRESS:	
		STREET 1:		ONE APPLE PARK WAY
		CITY:			CUP

In [21]:
import re

def extract_text_from_filing(file_path: str) -> dict:
    """
    Extracts readable text sections from a raw SEC filing.
    Strips XBRL/HTML tags and returns clean text for each key section.
    
    Output: dict with keys:
        - full_text: str (entire cleaned text, truncated to 500KB)
        - mda: str (Management Discussion & Analysis section)
        - risk_factors: str (Risk Factors section)
        - status: dict
    """
    
    result = {
        "full_text": None,
        "mda": None,
        "risk_factors": None,
        "status": {}
    }
    
    try:
        # Read file
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            raw = f.read()
        
        # --- Step 1: Strip HTML/XBRL tags ---
        # Remove script and style blocks first
        clean = re.sub(r'<script[^>]*>.*?</script>', ' ', raw, flags=re.DOTALL | re.IGNORECASE)
        clean = re.sub(r'<style[^>]*>.*?</style>', ' ', clean, flags=re.DOTALL | re.IGNORECASE)
        
        # Remove all remaining HTML tags
        clean = re.sub(r'<[^>]+>', ' ', clean)
        
        # Remove XBRL namespace declarations
        clean = re.sub(r'&[a-zA-Z]+;', ' ', clean)
        
        # Collapse multiple whitespace into single space
        clean = re.sub(r'\s+', ' ', clean)
        
        # Truncate to 500KB worth of characters (~500,000 chars)
        if len(clean) > 500_000:
            clean = clean[:500_000]
            result["status"]["truncated"] = True
        
        result["full_text"] = clean.strip()
        result["status"]["total_chars"] = len(result["full_text"])
        
        # --- Step 2: Extract MD&A section ---
        mda_pattern = re.search(
            r'(item\s*7[\.\s]*management.{0,50}discussion.{0,200})(.*?)(item\s*7a|item\s*8)',
            clean,
            flags=re.IGNORECASE | re.DOTALL
        )
        if mda_pattern:
            result["mda"] = mda_pattern.group(2)[:50_000]  # cap at 50k chars
            result["status"]["mda_found"] = True
        else:
            result["status"]["mda_found"] = False
        
        # --- Step 3: Extract Risk Factors section ---
        risk_pattern = re.search(
            r'(item\s*1a[\.\s]*risk\s*factors)(.*?)(item\s*1b|item\s*2)',
            clean,
            flags=re.IGNORECASE | re.DOTALL
        )
        if risk_pattern:
            result["risk_factors"] = risk_pattern.group(2)[:50_000]
            result["status"]["risk_factors_found"] = True
        else:
            result["status"]["risk_factors_found"] = False
        
        result["status"]["success"] = True
        
    except Exception as e:
        result["status"]["success"] = False
        result["status"]["error"] = str(e)
    
    return result


# Test on Apple's 10-K
file_path = "data/filings/sec-edgar-filings/AAPL/10-K/0000320193-25-000079/full-submission.txt"
extracted = extract_text_from_filing(file_path)

print(f"Status: {extracted['status']}")
print(f"\nFull text preview (first 500 chars):")
print(extracted["full_text"][:500] if extracted["full_text"] else "None")
print(f"\nMD&A preview (first 500 chars):")
print(extracted["mda"][:500] if extracted["mda"] else "None")
print(f"\nRisk Factors preview (first 500 chars):")
print(extracted["risk_factors"][:500] if extracted["risk_factors"] else "None")

Status: {'truncated': True, 'total_chars': 499999, 'mda_found': True, 'risk_factors_found': True, 'success': True}

Full text preview (first 500 chars):
0000320193-25-000079.txt : 20251031 0000320193-25-000079.hdr.sgml : 20251031 20251031060126 ACCESSION NUMBER: 0000320193-25-000079 CONFORMED SUBMISSION TYPE: 10-K PUBLIC DOCUMENT COUNT: 91 CONFORMED PERIOD OF REPORT: 20250927 FILED AS OF DATE: 20251031 DATE AS OF CHANGE: 20251031 FILER: COMPANY DATA: COMPANY CONFORMED NAME: Apple Inc. CENTRAL INDEX KEY: 0000320193 STANDARD INDUSTRIAL CLASSIFICATION: ELECTRONIC COMPUTERS [3571] ORGANIZATION NAME: 06 Technology EIN: 942404110 STATE OF INCORPORATIO

MD&A preview (first 500 chars):
Changes in and Disagreements with Accountants on Accounting and Financial Disclosure 52 Item 9A. Controls and Procedures 52 Item 9B. Other Information 53 Item 9C. Disclosure Regarding Foreign Jurisdictions that Prevent Inspections 53 Part III Item 10. Directors, Executive Officers and Corporate Governance 53 Ite

In [22]:
def extract_text_from_filing(file_path: str) -> dict:
    """
    Extracts readable text sections from a raw SEC filing.
    Strips XBRL/HTML tags and returns clean text for each key section.
    
    Output: dict with keys:
        - full_text: str (entire cleaned text, truncated to 500KB)
        - mda: str (Management Discussion & Analysis section)
        - risk_factors: str (Risk Factors section)
        - status: dict
    """
    
    result = {
        "full_text": None,
        "mda": None,
        "risk_factors": None,
        "status": {}
    }
    
    try:
        # Read file
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            raw = f.read()
        
        # --- Step 1: Strip HTML/XBRL tags ---
        clean = re.sub(r'<script[^>]*>.*?</script>', ' ', raw, flags=re.DOTALL | re.IGNORECASE)
        clean = re.sub(r'<style[^>]*>.*?</style>', ' ', clean, flags=re.DOTALL | re.IGNORECASE)
        clean = re.sub(r'<[^>]+>', ' ', clean)
        clean = re.sub(r'&[a-zA-Z]+;', ' ', clean)
        clean = re.sub(r'\s+', ' ', clean)
        
        # Truncate to 500KB
        if len(clean) > 500_000:
            clean = clean[:500_000]
            result["status"]["truncated"] = True
        
        result["full_text"] = clean.strip()
        result["status"]["total_chars"] = len(result["full_text"])
        
        # --- Step 2: Extract MD&A ---
        # Find ALL occurrences of Item 7
        # First occurrence is table of contents — we want the second
        mda_matches = list(re.finditer(
            r'item\s*7[\.\s]*management.{0,50}discussion',
            clean,
            flags=re.IGNORECASE
        ))
        
        if len(mda_matches) >= 2:
            # Use second occurrence — actual section not TOC
            start_pos = mda_matches[1].start()
            
            # Find end — next major item after MD&A
            end_match = re.search(
                r'item\s*7a|item\s*8',
                clean[start_pos + 100:],  # skip past the header itself
                flags=re.IGNORECASE
            )
            
            if end_match:
                end_pos = start_pos + 100 + end_match.start()
                result["mda"] = clean[start_pos:end_pos][:50_000]
            else:
                result["mda"] = clean[start_pos:start_pos + 50_000]
            
            result["status"]["mda_found"] = True
            
        elif len(mda_matches) == 1:
            # Only one occurrence — use it anyway
            start_pos = mda_matches[0].start()
            result["mda"] = clean[start_pos:start_pos + 50_000]
            result["status"]["mda_found"] = True
            result["status"]["mda_warning"] = "Only one Item 7 found — may be TOC"
        else:
            result["status"]["mda_found"] = False
        
        # --- Step 3: Extract Risk Factors ---
        # Same approach — find second occurrence
        risk_matches = list(re.finditer(
            r'item\s*1a[\.\s]*risk\s*factors',
            clean,
            flags=re.IGNORECASE
        ))
        
        if len(risk_matches) >= 2:
            start_pos = risk_matches[1].start()
            end_match = re.search(
                r'item\s*1b|item\s*2',
                clean[start_pos + 100:],
                flags=re.IGNORECASE
            )
            if end_match:
                end_pos = start_pos + 100 + end_match.start()
                result["risk_factors"] = clean[start_pos:end_pos][:50_000]
            else:
                result["risk_factors"] = clean[start_pos:start_pos + 50_000]
            result["status"]["risk_factors_found"] = True
            
        elif len(risk_matches) == 1:
            start_pos = risk_matches[0].start()
            result["risk_factors"] = clean[start_pos:start_pos + 50_000]
            result["status"]["risk_factors_found"] = True
            result["status"]["risk_warning"] = "Only one Item 1A found — may be TOC"
        else:
            result["status"]["risk_factors_found"] = False
        
        result["status"]["success"] = True
        
    except Exception as e:
        result["status"]["success"] = False
        result["status"]["error"] = str(e)
    
    return result


# Test again
file_path = "data/filings/sec-edgar-filings/AAPL/10-K/0000320193-25-000079/full-submission.txt"
extracted = extract_text_from_filing(file_path)

print(f"Status: {extracted['status']}")
print(f"\nMD&A preview (first 1000 chars):")
print(extracted["mda"][:1000] if extracted["mda"] else "None")
print(f"\nRisk Factors preview (first 1000 chars):")
print(extracted["risk_factors"][:1000] if extracted["risk_factors"] else "None")

Status: {'truncated': True, 'total_chars': 499999, 'mda_found': True, 'mda_warning': 'Only one Item 7 found — may be TOC', 'risk_factors_found': True, 'risk_warning': 'Only one Item 1A found — may be TOC', 'success': True}

MD&A preview (first 1000 chars):
Item 7. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operations 21 Item 7A. Quantitative and Qualitative Disclosures About Market Risk 27 Item 8. Financial Statements and Supplementary Data 28 Item 9. Changes in and Disagreements with Accountants on Accounting and Financial Disclosure 52 Item 9A. Controls and Procedures 52 Item 9B. Other Information 53 Item 9C. Disclosure Regarding Foreign Jurisdictions that Prevent Inspections 53 Part III Item 10. Directors, Executive Officers and Corporate Governance 53 Item 11. Executive Compensation 53 Item 12 . Security Ownership of Certain Beneficial Owners and Management and Related Stockholder Matters 53 Item 13 . Certain Relationships and Related Transacti

In [23]:
def extract_text_from_filing(file_path: str) -> dict:
    """
    Extracts readable text sections from a raw SEC filing.
    Searches entire file for key sections before truncating.
    """
    
    result = {
        "full_text": None,
        "mda": None,
        "risk_factors": None,
        "status": {}
    }
    
    try:
        # Read entire file
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            raw = f.read()
        
        # --- Step 1: Strip HTML/XBRL tags from FULL file first ---
        clean = re.sub(r'<script[^>]*>.*?</script>', ' ', raw, flags=re.DOTALL | re.IGNORECASE)
        clean = re.sub(r'<style[^>]*>.*?</style>', ' ', clean, flags=re.DOTALL | re.IGNORECASE)
        clean = re.sub(r'<[^>]+>', ' ', clean)
        clean = re.sub(r'&[a-zA-Z#0-9]+;', ' ', clean)
        clean = re.sub(r'\s+', ' ', clean)
        
        print(f"Full cleaned text length: {len(clean):,} chars")
        
        # --- Step 2: Find MD&A in FULL text ---
        mda_matches = list(re.finditer(
            r'item\s*7[\.\s]*management.{0,50}discussion',
            clean,
            flags=re.IGNORECASE
        ))
        print(f"Item 7 occurrences found: {len(mda_matches)}")
        for i, m in enumerate(mda_matches):
            print(f"  Match {i+1} at position {m.start():,}: ...{clean[m.start():m.start()+80]}...")
        
        # --- Step 3: Find Risk Factors in FULL text ---
        risk_matches = list(re.finditer(
            r'item\s*1a[\.\s]*risk\s*factors',
            clean,
            flags=re.IGNORECASE
        ))
        print(f"\nItem 1A occurrences found: {len(risk_matches)}")
        for i, m in enumerate(risk_matches):
            print(f"  Match {i+1} at position {m.start():,}: ...{clean[m.start():m.start()+80]}...")
            
    except Exception as e:
        result["status"]["success"] = False
        result["status"]["error"] = str(e)
    
    return result

# Test
file_path = "data/filings/sec-edgar-filings/AAPL/10-K/0000320193-25-000079/full-submission.txt"
extracted = extract_text_from_filing(file_path)

Full cleaned text length: 3,490,033 chars
Item 7 occurrences found: 2
  Match 1 at position 20,389: ...Item 7. Management s Discussion and Analysis of Financial Condition and Results ...
  Match 2 at position 118,736: ...Item 7. Management s Discussion and Analysis of Financial Condition and Results ...

Item 1A occurrences found: 2
  Match 1 at position 20,065: ...Item 1A. Risk Factors 5 Item 1B. Unresolved Staff Comments 17 Item 1C. Cybersecu...
  Match 2 at position 39,265: ...Item 1A. Risk Factors The following summarizes factors that could have a materia...


In [24]:
def extract_text_from_filing(file_path: str) -> dict:
    """
    Extracts readable text sections from a raw SEC filing.
    Searches entire file for key sections before truncating.
    
    Output: dict with keys:
        - full_text: str (first 500k chars of cleaned text)
        - mda: str (Management Discussion & Analysis)
        - risk_factors: str (Risk Factors section)
        - status: dict
    """
    
    result = {
        "full_text": None,
        "mda": None,
        "risk_factors": None,
        "status": {}
    }
    
    try:
        # Read entire file
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            raw = f.read()
        
        # --- Step 1: Strip HTML/XBRL tags from FULL file ---
        clean = re.sub(r'<script[^>]*>.*?</script>', ' ', raw, flags=re.DOTALL | re.IGNORECASE)
        clean = re.sub(r'<style[^>]*>.*?</style>', ' ', clean, flags=re.DOTALL | re.IGNORECASE)
        clean = re.sub(r'<[^>]+>', ' ', clean)
        clean = re.sub(r'&[a-zA-Z#0-9]+;', ' ', clean)
        clean = re.sub(r'\s+', ' ', clean)
        
        result["status"]["total_chars"] = len(clean)
        
        # --- Step 2: Extract MD&A ---
        # Use second occurrence (first is TOC)
        mda_matches = list(re.finditer(
            r'item\s*7[\.\s]*management.{0,50}discussion',
            clean,
            flags=re.IGNORECASE
        ))
        
        if len(mda_matches) >= 2:
            start_pos = mda_matches[1].start()
            # Find end of MD&A — Item 7A or Item 8
            end_match = re.search(
                r'item\s*7a|item\s*8',
                clean[start_pos + 200:],
                flags=re.IGNORECASE
            )
            if end_match:
                end_pos = start_pos + 200 + end_match.start()
                result["mda"] = clean[start_pos:end_pos][:50_000]
            else:
                result["mda"] = clean[start_pos:start_pos + 50_000]
            result["status"]["mda_found"] = True
            
        elif len(mda_matches) == 1:
            start_pos = mda_matches[0].start()
            result["mda"] = clean[start_pos:start_pos + 50_000]
            result["status"]["mda_found"] = True
            result["status"]["mda_warning"] = "Only one match — may be TOC"
        else:
            result["status"]["mda_found"] = False

        # --- Step 3: Extract Risk Factors ---
        risk_matches = list(re.finditer(
            r'item\s*1a[\.\s]*risk\s*factors',
            clean,
            flags=re.IGNORECASE
        ))
        
        if len(risk_matches) >= 2:
            start_pos = risk_matches[1].start()
            end_match = re.search(
                r'item\s*1b|item\s*2',
                clean[start_pos + 200:],
                flags=re.IGNORECASE
            )
            if end_match:
                end_pos = start_pos + 200 + end_match.start()
                result["risk_factors"] = clean[start_pos:end_pos][:50_000]
            else:
                result["risk_factors"] = clean[start_pos:start_pos + 50_000]
            result["status"]["risk_factors_found"] = True
            
        elif len(risk_matches) == 1:
            start_pos = risk_matches[0].start()
            result["risk_factors"] = clean[start_pos:start_pos + 50_000]
            result["status"]["risk_factors_found"] = True
            result["status"]["risk_warning"] = "Only one match — may be TOC"
        else:
            result["status"]["risk_factors_found"] = False

        # --- Step 4: Store truncated full text for general queries ---
        result["full_text"] = clean[:500_000]
        result["status"]["success"] = True
        
    except Exception as e:
        result["status"]["success"] = False
        result["status"]["error"] = str(e)
    
    return result


# Test
file_path = "data/filings/sec-edgar-filings/AAPL/10-K/0000320193-25-000079/full-submission.txt"
extracted = extract_text_from_filing(file_path)

print(f"Status: {extracted['status']}")
print(f"\nMD&A preview (first 1000 chars):")
print(extracted["mda"][:1000] if extracted["mda"] else "None")
print(f"\nRisk Factors preview (first 1000 chars):")
print(extracted["risk_factors"][:1000] if extracted["risk_factors"] else "None")

Status: {'total_chars': 3490033, 'mda_found': True, 'risk_factors_found': True, 'success': True}

MD&A preview (first 1000 chars):
Item 7. Management s Discussion and Analysis of Financial Condition and Results of Operations The following discussion should be read in conjunction with the consolidated financial statements and accompanying notes included in Part II, 

Risk Factors preview (first 1000 chars):
Item 1A. Risk Factors The following summarizes factors that could have a material adverse effect on the Company s business, reputation, results of operations, financial condition and stock price. The Company may not be able to accurately predict, control or mitigate these risks. Statements in this section are based on the Company s beliefs and opinions regarding matters that could materially adversely affect the Company in the future and are not representations as to whether such matters have or have not occurred previously. The risks and uncertainties described below are not exhaust

In [25]:
def get_filing(ticker: str, cik: str, user_email: str) -> dict:
    """
    Downloads most recent 10-K from EDGAR and extracts key sections.
    
    Input: 
        ticker (str) — e.g. "AAPL"
        cik (str) — e.g. "0000320193"
        user_email (str) — for EDGAR user agent
    Output: dict with keys:
        - ticker: str
        - file_path: str
        - full_text: str
        - mda: str
        - risk_factors: str
        - status: dict
    """
    
    import os
    from sec_edgar_downloader import Downloader
    
    result = {
        "ticker": ticker.upper(),
        "file_path": None,
        "full_text": None,
        "mda": None,
        "risk_factors": None,
        "status": {}
    }
    
    # --- Step 1: Download filing ---
    try:
        filing_dir = "data/filings"
        dl = Downloader("credit-committee", user_email, filing_dir)
        dl.get("10-K", ticker, limit=1)
        result["status"]["download"] = "success"
    except Exception as e:
        result["status"]["success"] = False
        result["status"]["error"] = f"Download failed: {str(e)}"
        return result
    
    # --- Step 2: Find the downloaded file ---
    try:
        ticker_dir = os.path.join(
            filing_dir, "sec-edgar-filings", 
            ticker.upper(), "10-K"
        )
        
        # Get most recent filing folder
        filing_folders = sorted(os.listdir(ticker_dir), reverse=True)
        
        if not filing_folders:
            result["status"]["success"] = False
            result["status"]["error"] = "No filing folders found after download"
            return result
        
        latest_folder = os.path.join(ticker_dir, filing_folders[0])
        file_path = os.path.join(latest_folder, "full-submission.txt")
        
        if not os.path.exists(file_path):
            result["status"]["success"] = False
            result["status"]["error"] = f"full-submission.txt not found in {latest_folder}"
            return result
        
        result["file_path"] = file_path
        result["status"]["file_found"] = True
        
    except Exception as e:
        result["status"]["success"] = False
        result["status"]["error"] = f"File search failed: {str(e)}"
        return result
    
    # --- Step 3: Extract text sections ---
    extracted = extract_text_from_filing(file_path)
    
    if not extracted["status"]["success"]:
        result["status"]["success"] = False
        result["status"]["error"] = extracted["status"].get("error", "Extraction failed")
        return result
    
    result["full_text"]    = extracted["full_text"]
    result["mda"]          = extracted["mda"]
    result["risk_factors"] = extracted["risk_factors"]
    result["status"].update(extracted["status"])
    result["status"]["success"] = True
    
    return result


# Test full pipeline
import os
USER_EMAIL = "YOUR_ACTUAL_EMAIL@gmail.com"

# AAPL already downloaded so this will be fast
data = get_filing("AAPL", "0000320193", USER_EMAIL)

print(f"Status: {data['status']}")
print(f"File path: {data['file_path']}")
print(f"\nMD&A length: {len(data['mda']):,} chars" if data['mda'] else "MD&A: None")
print(f"Risk Factors length: {len(data['risk_factors']):,} chars" if data['risk_factors'] else "Risk Factors: None")
print(f"\nMD&A preview:")
print(data['mda'][:300] if data['mda'] else "None")

Status: {'download': 'success', 'file_found': True, 'total_chars': 3490033, 'mda_found': True, 'risk_factors_found': True, 'success': True}
File path: data/filings\sec-edgar-filings\AAPL\10-K\0000320193-25-000079\full-submission.txt

MD&A length: 236 chars
Risk Factors length: 50,000 chars

MD&A preview:
Item 7. Management s Discussion and Analysis of Financial Condition and Results of Operations The following discussion should be read in conjunction with the consolidated financial statements and accompanying notes included in Part II, 


In [26]:
# Debug — let's see what's around position 118,736 in the full text
file_path = "data/filings/sec-edgar-filings/AAPL/10-K/0000320193-25-000079/full-submission.txt"

with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    raw = f.read()

import re
clean = re.sub(r'<script[^>]*>.*?</script>', ' ', raw, flags=re.DOTALL | re.IGNORECASE)
clean = re.sub(r'<style[^>]*>.*?</style>', ' ', clean, flags=re.DOTALL | re.IGNORECASE)
clean = re.sub(r'<[^>]+>', ' ', clean)
clean = re.sub(r'&[a-zA-Z#0-9]+;', ' ', clean)
clean = re.sub(r'\s+', ' ', clean)

# Find Item 7 matches
mda_matches = list(re.finditer(
    r'item\s*7[\.\s]*management.{0,50}discussion',
    clean,
    flags=re.IGNORECASE
))

# Show 3000 chars after second match to understand structure
start = mda_matches[1].start()
print(f"Second Item 7 starts at: {start:,}")
print(f"\nContent from position {start:,} to {start+3000:,}:")
print(clean[start:start+3000])

Second Item 7 starts at: 118,736

Content from position 118,736 to 121,736:
Item 7. Management s Discussion and Analysis of Financial Condition and Results of Operations The following discussion should be read in conjunction with the consolidated financial statements and accompanying notes included in Part II, Item 8 of this Form 10-K. This Item generally discusses 2025 and 2024 items and year-to-year comparisons between 2025 and 2024. Discussions of 2023 items and year-to-year comparisons between 2024 and 2023 are not included, and can be found in Management s Discussion and Analysis of Financial Condition and Results of Operations in Part II, Item 7 of the Company s Annual Report on Form 10-K for the fiscal year ended September 28, 2024. Product, Service and Software Announcements The Company announces new product, service and software offerings at various times during the year. Significant announcements during fiscal year 2025 included the following: First Quarter 2025: MacBook Pro 

In [27]:
def extract_text_from_filing(file_path: str) -> dict:
    """
    Extracts readable text sections from a raw SEC filing.
    Searches entire file for key sections before truncating.
    
    Output: dict with keys:
        - full_text: str (first 500k chars of cleaned text)
        - mda: str (Management Discussion & Analysis)
        - risk_factors: str (Risk Factors section)
        - status: dict
    """
    
    result = {
        "full_text": None,
        "mda": None,
        "risk_factors": None,
        "status": {}
    }
    
    try:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            raw = f.read()
        
        # --- Strip HTML/XBRL tags ---
        clean = re.sub(r'<script[^>]*>.*?</script>', ' ', raw, flags=re.DOTALL | re.IGNORECASE)
        clean = re.sub(r'<style[^>]*>.*?</style>', ' ', clean, flags=re.DOTALL | re.IGNORECASE)
        clean = re.sub(r'<[^>]+>', ' ', clean)
        clean = re.sub(r'&[a-zA-Z#0-9]+;', ' ', clean)
        clean = re.sub(r'\s+', ' ', clean)
        
        result["status"]["total_chars"] = len(clean)
        
        # --- Extract MD&A ---
        mda_matches = list(re.finditer(
            r'item\s*7[\.\s]*management.{0,50}discussion',
            clean,
            flags=re.IGNORECASE
        ))
        
        if len(mda_matches) >= 2:
            start_pos = mda_matches[1].start()
            
            # Look for Item 7A or Item 8 as a SECTION HEADER
            # Must be followed by a title word, not just a reference
            # Search from 5000 chars in to skip internal references
            search_from = start_pos + 5_000
            end_match = re.search(
                r'item\s*7a[\.\s]*quantitative|item\s*8[\.\s]*financial\s*statements',
                clean[search_from:],
                flags=re.IGNORECASE
            )
            
            if end_match:
                end_pos = search_from + end_match.start()
                result["mda"] = clean[start_pos:end_pos][:80_000]
            else:
                result["mda"] = clean[start_pos:start_pos + 80_000]
            
            result["status"]["mda_found"] = True
            
        elif len(mda_matches) == 1:
            start_pos = mda_matches[0].start()
            result["mda"] = clean[start_pos:start_pos + 80_000]
            result["status"]["mda_found"] = True
            result["status"]["mda_warning"] = "Only one match — may be TOC"
        else:
            result["status"]["mda_found"] = False

        # --- Extract Risk Factors ---
        risk_matches = list(re.finditer(
            r'item\s*1a[\.\s]*risk\s*factors',
            clean,
            flags=re.IGNORECASE
        ))
        
        if len(risk_matches) >= 2:
            start_pos = risk_matches[1].start()
            search_from = start_pos + 5_000
            end_match = re.search(
                r'item\s*1b[\.\s]*unresolved|item\s*2[\.\s]*properties',
                clean[search_from:],
                flags=re.IGNORECASE
            )
            if end_match:
                end_pos = search_from + end_match.start()
                result["risk_factors"] = clean[start_pos:end_pos][:80_000]
            else:
                result["risk_factors"] = clean[start_pos:start_pos + 80_000]
            result["status"]["risk_factors_found"] = True
            
        elif len(risk_matches) == 1:
            start_pos = risk_matches[0].start()
            result["risk_factors"] = clean[start_pos:start_pos + 80_000]
            result["status"]["risk_factors_found"] = True
            result["status"]["risk_warning"] = "Only one match — may be TOC"
        else:
            result["status"]["risk_factors_found"] = False

        # Store truncated full text
        result["full_text"] = clean[:500_000]
        result["status"]["success"] = True
        
    except Exception as e:
        result["status"]["success"] = False
        result["status"]["error"] = str(e)
    
    return result


# Test
file_path = "data/filings/sec-edgar-filings/AAPL/10-K/0000320193-25-000079/full-submission.txt"
extracted = extract_text_from_filing(file_path)

print(f"Status: {extracted['status']}")
print(f"MD&A length: {len(extracted['mda']):,} chars" if extracted['mda'] else "MD&A: None")
print(f"Risk Factors length: {len(extracted['risk_factors']):,} chars" if extracted['risk_factors'] else "Risk Factors: None")
print(f"\nMD&A preview (first 500 chars):")
print(extracted["mda"][:500] if extracted["mda"] else "None")
print(f"\nMD&A last 300 chars:")
print(extracted["mda"][-300:] if extracted["mda"] else "None")

Status: {'total_chars': 3490033, 'mda_found': True, 'risk_factors_found': True, 'success': True}
MD&A length: 17,981 chars
Risk Factors length: 68,021 chars

MD&A preview (first 500 chars):
Item 7. Management s Discussion and Analysis of Financial Condition and Results of Operations The following discussion should be read in conjunction with the consolidated financial statements and accompanying notes included in Part II, Item 8 of this Form 10-K. This Item generally discusses 2025 and 2024 items and year-to-year comparisons between 2025 and 2024. Discussions of 2023 items and year-to-year comparisons between 2024 and 2023 are not included, and can be found in Management s Discussi

MD&A last 300 chars:
ble a loss has been incurred and the amount is reasonably estimable, the determination of which requires significant judgment. Resolution of legal matters in a manner inconsistent with management s expectations could have a material impact on the Company s financial condition and operat

In [28]:
# Test Motley Fool transcript scraping
import requests
from bs4 import BeautifulSoup

ticker = "AAPL"

# Motley Fool earnings transcript URL pattern
url = f"https://www.fool.com/earnings-call-transcripts/?symbols={ticker}"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

try:
    response = requests.get(url, headers=headers, timeout=10)
    print(f"Status code: {response.status_code}")
    print(f"Response length: {len(response.text):,} chars")
    print(f"\nPreview:")
    print(response.text[:2000])
except Exception as e:
    print(f"Error: {e}")

Status code: 200
Response length: 283,319 chars

Preview:
<!DOCTYPE html><html lang="en" class="__className_809ffe"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1"/><link rel="preload" as="image" imageSrcSet="https://g.foolcdn.com/misc-assets/logo-tmf-primary-1-magenta-purple-reversed.svg?_w=256 1x, https://g.foolcdn.com/misc-assets/logo-tmf-primary-1-magenta-purple-reversed.svg?_w=320 2x"/><link rel="preload" as="image" imageSrcSet="https://g.foolcdn.com/image/?url=https%3A%2F%2Fg.foolcdn.com%2Fmisc-assets%2Ftmf_holdingco_logo_stacked_magenta_royalpurple_rev.png&amp;w=48&amp;op=resize 1x, https://g.foolcdn.com/image/?url=https%3A%2F%2Fg.foolcdn.com%2Fmisc-assets%2Ftmf_holdingco_logo_stacked_magenta_royalpurple_rev.png&amp;w=96&amp;op=resize 2x"/><link rel="stylesheet" href="https://g.foolcdn.com/static/foolcom/_next/static/css/2b9a8dda186f654e.css" data-precedence="next"/><link rel="stylesheet" href="https://g.foolcdn.com/static/foolcom

In [29]:
# Try direct search for AAPL transcript on Motley Fool
import requests
from bs4 import BeautifulSoup

# Motley Fool transcripts follow a predictable URL pattern
# Let's search Google for the most recent one
url = "https://www.fool.com/search/solr.aspx?q=apple+earnings+call+transcript+2025&type=13"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

try:
    response = requests.get(url, headers=headers, timeout=10)
    print(f"Status code: {response.status_code}")
    soup = BeautifulSoup(response.text, "html.parser")
    
    # Look for transcript links
    links = soup.find_all("a", href=True)
    transcript_links = [
        l["href"] for l in links 
        if "transcript" in l["href"].lower() 
        and "apple" in l.get_text().lower()
    ]
    
    print(f"\nTranscript links found: {len(transcript_links)}")
    for link in transcript_links[:5]:
        print(link)
        
except Exception as e:
    print(f"Error: {e}")

Status code: 410

Transcript links found: 0


In [30]:
# Try fetching a known Motley Fool transcript URL directly
# Motley Fool transcript URLs follow this pattern:
# https://www.fool.com/earnings/call-transcripts/YEAR/MONTH/DAY/company-name-qX-YEAR-earnings-call-transcript/

import requests
from bs4 import BeautifulSoup

# Known recent Apple transcript URL pattern
urls_to_try = [
    "https://www.fool.com/earnings/call-transcripts/2025/07/31/apple-q3-2025-earnings-call-transcript/",
    "https://www.fool.com/earnings/call-transcripts/2025/05/01/apple-q2-2025-earnings-call-transcript/",
    "https://www.fool.com/earnings/call-transcripts/2025/01/30/apple-q1-2025-earnings-call-transcript/",
]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

for url in urls_to_try:
    try:
        response = requests.get(url, headers=headers, timeout=10)
        print(f"URL: {url}")
        print(f"Status: {response.status_code}")
        if response.status_code == 200:
            print("SUCCESS — found transcript page")
            print(f"Response length: {len(response.text):,} chars")
            break
        print()
    except Exception as e:
        print(f"Error: {e}")

URL: https://www.fool.com/earnings/call-transcripts/2025/07/31/apple-q3-2025-earnings-call-transcript/
Status: 404

URL: https://www.fool.com/earnings/call-transcripts/2025/05/01/apple-q2-2025-earnings-call-transcript/
Status: 404

URL: https://www.fool.com/earnings/call-transcripts/2025/01/30/apple-q1-2025-earnings-call-transcript/
Status: 404



In [31]:
# Try alternative transcript sources
import requests
from bs4 import BeautifulSoup

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# Option 1 — Try Motley Fool new URL structure
urls_to_try = [
    "https://www.fool.com/earnings/call-transcripts/2025/07/31/aapl-q3-2025-earnings-call-transcript/",
    "https://www.fool.com/earnings/call-transcripts/2025/05/01/aapl-q2-2025-earnings-call-transcript/",
    # Option 2 — Try Seeking Alpha (free articles)
    "https://seekingalpha.com/symbol/AAPL/earnings/transcripts",
    # Option 3 — Try stockanalysis.com
    "https://stockanalysis.com/stocks/aapl/transcripts/",
]

for url in urls_to_try:
    try:
        response = requests.get(url, headers=headers, timeout=10)
        print(f"Status {response.status_code}: {url}")
        if response.status_code == 200:
            print(f"  Response length: {len(response.text):,} chars")
            soup = BeautifulSoup(response.text, "html.parser")
            # Get page title to confirm what we got
            title = soup.find("title")
            print(f"  Page title: {title.text if title else 'No title'}")
    except Exception as e:
        print(f"Error on {url}: {e}")

Status 404: https://www.fool.com/earnings/call-transcripts/2025/07/31/aapl-q3-2025-earnings-call-transcript/
Status 404: https://www.fool.com/earnings/call-transcripts/2025/05/01/aapl-q2-2025-earnings-call-transcript/
Status 403: https://seekingalpha.com/symbol/AAPL/earnings/transcripts
Status 200: https://stockanalysis.com/stocks/aapl/transcripts/
  Response length: 348,102 chars
  Page title: Apple (AAPL) Earnings Call Transcripts


In [32]:
# Parse stockanalysis.com transcripts page
url = "https://stockanalysis.com/stocks/aapl/transcripts/"

response = requests.get(url, headers=headers, timeout=10)
soup = BeautifulSoup(response.text, "html.parser")

# Find all links that look like transcript links
links = soup.find_all("a", href=True)
transcript_links = [
    l["href"] for l in links 
    if "transcript" in l["href"].lower()
]

print(f"Transcript links found: {len(transcript_links)}")
for link in transcript_links[:10]:
    print(link)
    

Transcript links found: 148
/stocks/aapl/transcripts/
/stocks/aapl/transcripts/548666-q2-2026/
/stocks/aapl/transcripts/548666-q2-2026/
/stocks/aapl/transcripts/406161-q1-2026/
/stocks/aapl/transcripts/406161-q1-2026/
/stocks/aapl/transcripts/369370-q4-2025/
/stocks/aapl/transcripts/369370-q4-2025/
/stocks/aapl/transcripts/382915-product-launch/
/stocks/aapl/transcripts/382915-product-launch/
/stocks/aapl/transcripts/341149-q3-2025/


In [33]:
# Fetch the most recent transcript
base_url = "https://stockanalysis.com"
most_recent = "/stocks/aapl/transcripts/548666-q2-2026/"

transcript_url = base_url + most_recent

response = requests.get(transcript_url, headers=headers, timeout=10)
print(f"Status: {response.status_code}")
print(f"Length: {len(response.text):,} chars")

soup = BeautifulSoup(response.text, "html.parser")

# Get page title
title = soup.find("title")
print(f"Title: {title.text if title else 'No title'}")

# Try to find the transcript body
# Look for the main content area
content_candidates = [
    soup.find("div", {"class": lambda x: x and "transcript" in x.lower() if x else False}),
    soup.find("article"),
    soup.find("main"),
    soup.find("div", {"id": "main"}),
]

for i, candidate in enumerate(content_candidates):
    if candidate:
        text = candidate.get_text(separator=" ", strip=True)
        print(f"\nCandidate {i+1} ({len(text):,} chars):")
        print(text[:500])
        break

Status: 400
Length: 38 chars
Title: No title


In [34]:
# Try with full browser headers
headers_full = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "none",
    "Sec-Fetch-User": "?1",
    "Referer": "https://stockanalysis.com/stocks/aapl/transcripts/"
}

transcript_url = "https://stockanalysis.com/stocks/aapl/transcripts/548666-q2-2026/"

try:
    # Use a session to maintain cookies
    session = requests.Session()
    
    # First visit the transcripts index page
    session.get(
        "https://stockanalysis.com/stocks/aapl/transcripts/",
        headers=headers_full,
        timeout=10
    )
    
    # Then fetch the specific transcript
    response = session.get(
        transcript_url,
        headers=headers_full,
        timeout=10
    )
    
    print(f"Status: {response.status_code}")
    print(f"Length: {len(response.text):,} chars")
    print(f"\nPreview:")
    print(response.text[:1000])
    
except Exception as e:
    print(f"Error: {e}")

Status: 200
Length: 53,285 chars

Preview:
e øÿ þ7Q«ÎÎåôÃÙ;ÓI©-v;.§ãi»â*»SÛåÍËÈO °RS×=ïí6÷9×2ÿ3ÍïNºÀÇ9rèdEVb(p`äNê¼ýowìÞ
üqý¶óm:c²¦ys=&kÀ(7ì2tYóæâzDÕ5o..®-&í¨BÄ´ËæÔ:uóæââÚbRàÅ]¶h<N>¤Zïº´ËºKã®ÃE·HºKã
´ÓI+Cb«îxd(ðæââÚhw1`¿ËÖ_Õ4­µµsR{k#¦¸fôGüúýî¾´1fÅt6GÄ5ïoânæNû}{ ÿ9¹÷âwQ¾)ýúæÞßèÿ[Ü}úyÿÓ?¿`mÐS?bRÚ<¨³-þY¿_|
óïë@ÍöÐû 7Ód®nnÞ­à¨­·]Rá}ð^´[ïpu{÷éÝ¤õ·¸¦D¿Å¬¹^Çl Ú7÷nÁ´nGågvpÔ®óGdÍÿõ_ðúåÍÅÅ«oNgIuLÚ×¶Þ®cRI·ëYeqôú-!_*Aë"A6Ña&hï²ïèÃlü,@0Qªà´"´ÊH
Jý'pÁØ©ÌÜýqé<=ÎÏ÷Q'¼àQ'üê½ªà9ùöpª¸¨ÁÓQ§aÛªÐ]·I­UáüÕ¨0à×¹9)Ú­ÌØ×RÎèti¯B¾!±½Ñ§¡©ÝDÛLTªêm~k}é<á.SÓdt«önmº?}ÞeÍïÙ_!bÁ)eÛ¤


In [35]:
import requests
from bs4 import BeautifulSoup

headers_full = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Referer": "https://stockanalysis.com/stocks/aapl/transcripts/"
}

# Remove Accept-Encoding so requests handles decompression automatically
session = requests.Session()

session.get(
    "https://stockanalysis.com/stocks/aapl/transcripts/",
    headers=headers_full,
    timeout=10
)

response = session.get(
    "https://stockanalysis.com/stocks/aapl/transcripts/548666-q2-2026/",
    headers=headers_full,
    timeout=10
)

print(f"Status: {response.status_code}")
print(f"Encoding: {response.encoding}")
print(f"Length: {len(response.text):,} chars")
print(f"\nPreview:")
print(response.text[:2000])

Status: 200
Encoding: ISO-8859-1
Length: 326,683 chars

Preview:
<!doctype html>
<html lang="en">
	<head>
		<meta charset="utf-8" />
		<meta name="viewport" content="width=device-width, initial-scale=1" />
		
		<link href="/_app/immutable/assets/0.De_zEI4s.css" rel="stylesheet">
		<link href="/_app/immutable/assets/AudioDock.HYpnB2UE.css" rel="stylesheet">
		<link href="/_app/immutable/assets/_TrustBlock.DX2EOQbN.css" rel="stylesheet">
		<link href="/_app/immutable/assets/_TranscriptsDetailLayout.4Y7hZrRo.css" rel="stylesheet"><!--[--><!--[--><script async src="https://securepubads.g.doubleclick.net/tag/js/gpt.js"></script> <script>
			InvestingChannelQueue = window.InvestingChannelQueue || []
		</script> <script async src="https://u5.investingchannel.com/static/uat.js"></script><!--]--><!--]--><!--[--><!--[--><meta name="description" content="Full Q2 2026 earnings call transcript and audio for Apple (AAPL), with commentary from Tim Cook (CEO), John Ternus (Senior Vice President, Hardw

In [36]:
soup = BeautifulSoup(response.text, "html.parser")

# Remove script and style tags
for tag in soup(["script", "style"]):
    tag.decompose()

# Try finding the transcript content
# Look for the main content div
candidates = [
    soup.find("div", {"class": lambda x: x and "transcript" in x.lower() if x else False}),
    soup.find("div", {"class": lambda x: x and "content" in x.lower() if x else False}),
    soup.find("article"),
    soup.find("main"),
]

for i, candidate in enumerate(candidates):
    if candidate:
        text = candidate.get_text(separator=" ", strip=True)
        print(f"Candidate {i+1} — {len(text):,} chars")
        print(text[:1000])
        print("---")
        break

# Also try getting all paragraph tags
print("\n\nParagraph approach:")
paragraphs = soup.find_all("p")
print(f"Total paragraphs found: {len(paragraphs)}")
if paragraphs:
    all_text = " ".join([p.get_text(strip=True) for p in paragraphs])
    print(f"Combined paragraph text: {len(all_text):,} chars")
    print(all_text[:1000])

Candidate 4 — 55,814 chars
Apple Inc. (AAPL) NASDAQ: AAPL Â· Real-Time Price Â· USD Full Chart Watchlist Alerts Compare 308.63 +14.25 (4.84%) At close: Jul 2, 2026, 4:00 PM EDT 308.44 -0.19 (-0.06%) After-hours: Jul 2, 2026, 7:59 PM EDT Overview Financials Forecast Statistics Metrics Dividends History Profile Chart Profile Transcripts Filings Employees â View all transcripts Earnings Call: Q2 2026 Apr 30, 2026 Full Transcript Full Summary Earnings release Quarterly report Play Audio Download Transcript Full Transcript Full Summary Summary Revenue grew 17% year-over-year to $111.2B, with net income up 22% and record EPS. Strong double-digit growth was seen across all segments and geographies, despite supply constraints. CEO transition and expanded capital return program announced. Suhasini Chandramouli Director of Investor Relations, Apple Good afternoon, welcome to the Apple Q2 fiscal year 2026 earnings conference call. My name is Suhasini Chandramouli, Director of Investor Relations

In [37]:
def get_transcript(ticker: str) -> dict:
    """
    Scrapes most recent earnings call transcript from stockanalysis.com
    
    Input: ticker (str) — e.g. "AAPL"
    Output: dict with keys:
        - ticker: str
        - transcript_text: str
        - quarter: str (e.g. "Q2 2026")
        - status: dict
    """
    
    result = {
        "ticker": ticker.upper(),
        "transcript_text": None,
        "quarter": None,
        "status": {}
    }
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
    }
    
    base_url = "https://stockanalysis.com"
    index_url = f"{base_url}/stocks/{ticker.lower()}/transcripts/"
    
    try:
        # --- Step 1: Get transcripts index page ---
        session = requests.Session()
        index_response = session.get(
            index_url,
            headers=headers,
            timeout=10
        )
        
        if index_response.status_code != 200:
            result["status"]["success"] = False
            result["status"]["error"] = (
                f"Transcripts index page returned "
                f"{index_response.status_code}"
            )
            return result
        
        # --- Step 2: Find most recent transcript link ---
        soup = BeautifulSoup(index_response.text, "html.parser")
        links = soup.find_all("a", href=True)
        
        transcript_links = [
            l["href"] for l in links
            if "transcript" in l["href"].lower()
            and ticker.lower() in l["href"].lower()
            and l["href"] != f"/stocks/{ticker.lower()}/transcripts/"
            and "product-launch" not in l["href"].lower()
        ]
        
        # Deduplicate while preserving order
        seen = set()
        unique_links = []
        for link in transcript_links:
            if link not in seen:
                seen.add(link)
                unique_links.append(link)
        
        if not unique_links:
            result["status"]["success"] = False
            result["status"]["error"] = "No transcript links found"
            return result
        
        # Most recent is first
        most_recent_link = unique_links[0]
        
        # Extract quarter from URL e.g. "548666-q2-2026" -> "Q2 2026"
        import re
        quarter_match = re.search(
            r'(\d+)-(q\d+)-(\d{4})',
            most_recent_link,
            flags=re.IGNORECASE
        )
        if quarter_match:
            result["quarter"] = (
                f"{quarter_match.group(2).upper()} "
                f"{quarter_match.group(3)}"
            )
        
        # --- Step 3: Fetch transcript page ---
        transcript_url = base_url + most_recent_link
        result["status"]["transcript_url"] = transcript_url
        
        transcript_response = session.get(
            transcript_url,
            headers={
                **headers,
                "Referer": index_url
            },
            timeout=10
        )
        
        if transcript_response.status_code != 200:
            result["status"]["success"] = False
            result["status"]["error"] = (
                f"Transcript page returned "
                f"{transcript_response.status_code}"
            )
            return result
        
        # --- Step 4: Extract text ---
        soup = BeautifulSoup(transcript_response.text, "html.parser")
        
        # Remove script and style tags
        for tag in soup(["script", "style"]):
            tag.decompose()
        
        # Extract paragraph text
        paragraphs = soup.find_all("p")
        transcript_text = " ".join([
            p.get_text(strip=True) 
            for p in paragraphs 
            if len(p.get_text(strip=True)) > 20
        ])
        
        if not transcript_text or len(transcript_text) < 500:
            result["status"]["success"] = False
            result["status"]["error"] = "Transcript text too short — likely blocked"
            return result
        
        result["transcript_text"] = transcript_text[:50_000]
        result["status"]["char_count"] = len(transcript_text)
        result["status"]["success"] = True
        
    except Exception as e:
        result["status"]["success"] = False
        result["status"]["error"] = str(e)
    
    return result


# Test on three tickers
for ticker in ["AAPL", "BA", "NVDA"]:
    print(f"\n{'='*40}")
    print(f"Testing: {ticker}")
    data = get_transcript(ticker)
    if data["status"]["success"]:
        print(f"Quarter: {data['quarter']}")
        print(f"URL: {data['status']['transcript_url']}")
        print(f"Length: {data['status']['char_count']:,} chars")
        print(f"Preview: {data['transcript_text'][:200]}")
    else:
        print(f"FAILED: {data['status']['error']}")


Testing: AAPL
Quarter: Q2 2026
URL: https://stockanalysis.com/stocks/aapl/transcripts/548666-q2-2026/
Length: 51,669 chars
Preview: Revenue grew 17% year-over-year to $111.2B, with net income up 22% and record EPS. Strong double-digit growth was seen across all segments and geographies, despite supply constraints. CEO transition a

Testing: BA
Quarter: None
URL: https://stockanalysis.com/stocks/ba/transcripts/671085-bernstein-42nd-annual-strategic-decisions-conference/
Length: 38,744 chars
Preview: Production and backlog remain strong, with commercial and defense portfolios at record levels. Key milestones ahead include 737-7/-10 and 777X certifications, production ramp to 47 and 52 per month, a

Testing: NVDA
Quarter: Q1 2027
URL: https://stockanalysis.com/stocks/nvda/transcripts/568907-q1-2027/
Length: 45,285 chars
Preview: Record quarterly revenue and free cash flow driven by surging AI infrastructure demand, with Data Center and Edge Computing segments both posting strong growth. 

In [38]:
# Test FRED API
from fredapi import Fred
import os
from dotenv import load_dotenv

load_dotenv()

FRED_API_KEY = os.getenv("FRED_API_KEY")
fred = Fred(api_key=FRED_API_KEY)

# Test pulling one series first
gdp = fred.get_series("GDP")
print(f"GDP series length: {len(gdp)}")
print(f"Most recent value: {gdp.iloc[-1]}")
print(f"Most recent date: {gdp.index[-1]}")
print(f"\nLast 4 quarters:")
print(gdp.tail(4))

GDP series length: 321
Most recent value: 31865.721
Most recent date: 2026-01-01 00:00:00

Last 4 quarters:
2025-04-01    30485.729
2025-07-01    31098.027
2025-10-01    31422.526
2026-01-01    31865.721
dtype: float64


In [39]:
import pandas as pd

# Pull all five macro series
series_map = {
    "GDP":     "GDP — Gross Domestic Product (Quarterly, Billions USD)",
    "UNRATE":  "Unemployment Rate (Monthly, %)",
    "FEDFUNDS": "Federal Funds Rate (Monthly, %)",
    "BAA10Y":  "Moody's BAA Corporate Bond Spread (Monthly, %)",
    "T10Y2Y":  "10Y minus 2Y Treasury Spread (Daily, %)"
}

macro_data = {}

for series_id, description in series_map.items():
    try:
        data = fred.get_series(series_id)
        # Get last 8 quarters worth of data
        # For monthly/daily series, resample to quarterly
        if series_id == "GDP":
            recent = data.tail(8)
        else:
            # Resample to quarterly average for non-GDP series
            recent = data.resample("QE").mean().tail(8)
        
        macro_data[series_id] = {
            "description": description,
            "values": recent,
            "latest_value": round(float(recent.iloc[-1]), 2),
            "latest_date": str(recent.index[-1].date())
        }
        print(f"{series_id}: latest = {macro_data[series_id]['latest_value']} ({macro_data[series_id]['latest_date']})")
    except Exception as e:
        print(f"{series_id}: FAILED — {e}")
        macro_data[series_id] = None

print("\nAll series pulled successfully:", all(v is not None for v in macro_data.values()))

GDP: latest = 31865.72 (2026-01-01)
UNRATE: latest = 4.27 (2026-06-30)
FEDFUNDS: latest = 3.63 (2026-06-30)
BAA10Y: latest = 1.54 (2026-09-30)
T10Y2Y: latest = 0.33 (2026-09-30)

All series pulled successfully: True


In [40]:
def get_macro_context() -> dict:
    """
    Pulls five macro series from FRED and returns current snapshot
    plus 8-quarter history for each.

    Output: dict with keys:
        - snapshot: dict (latest value for each series)
        - history: dict (8-quarter DataFrame for each series)
        - narrative: str (plain English summary for LLM)
        - status: dict
    """

    result = {
        "snapshot": {},
        "history": {},
        "narrative": None,
        "status": {}
    }

    series_map = {
        "GDP":      "Gross Domestic Product (Billions USD)",
        "UNRATE":   "Unemployment Rate (%)",
        "FEDFUNDS": "Federal Funds Rate (%)",
        "BAA10Y":   "Moody's BAA Corporate Bond Spread (%)",
        "T10Y2Y":   "10Y minus 2Y Treasury Spread (%)"
    }

    failed = []

    for series_id, description in series_map.items():
        try:
            data = fred.get_series(series_id)

            if series_id == "GDP":
                recent = data.tail(8)
            else:
                recent = data.resample("QE").mean().tail(8)

            latest_val  = round(float(recent.iloc[-1]), 2)
            latest_date = str(recent.index[-1].date())

            # Quarter over quarter change
            if len(recent) >= 2:
                prev_val = float(recent.iloc[-2])
                qoq_change = round(latest_val - prev_val, 2)
                direction = "up" if qoq_change > 0 else "down"
            else:
                qoq_change = None
                direction = "unknown"

            result["snapshot"][series_id] = {
                "description": description,
                "latest_value": latest_val,
                "latest_date": latest_date,
                "qoq_change": qoq_change,
                "direction": direction
            }

            result["history"][series_id] = recent

        except Exception as e:
            failed.append(series_id)
            result["snapshot"][series_id] = None

    # --- Build plain English narrative for LLM ---
    try:
        s = result["snapshot"]

        gdp       = s.get("GDP", {})
        unrate    = s.get("UNRATE", {})
        fedfunds  = s.get("FEDFUNDS", {})
        baa10y    = s.get("BAA10Y", {})
        t10y2y    = s.get("T10Y2Y", {})

        narrative_parts = []

        if gdp:
            narrative_parts.append(
                f"GDP stood at ${gdp['latest_value']:,}bn as of "
                f"{gdp['latest_date']}, "
                f"{'expanding' if gdp['direction'] == 'up' else 'contracting'} "
                f"by ${abs(gdp['qoq_change']):,}bn quarter over quarter."
            )

        if unrate:
            narrative_parts.append(
                f"The unemployment rate is {unrate['latest_value']}%, "
                f"{'rising' if unrate['direction'] == 'up' else 'falling'} "
                f"{abs(unrate['qoq_change'])}pp quarter over quarter."
            )

        if fedfunds:
            narrative_parts.append(
                f"The Federal Funds Rate stands at {fedfunds['latest_value']}%, "
                f"indicating a "
                f"{'tight' if fedfunds['latest_value'] > 4 else 'moderate' if fedfunds['latest_value'] > 2 else 'loose'} "
                f"monetary policy environment."
            )

        if baa10y:
            narrative_parts.append(
                f"The Moody's BAA corporate bond spread is {baa10y['latest_value']}%, "
                f"suggesting "
                f"{'elevated' if baa10y['latest_value'] > 3 else 'moderate' if baa10y['latest_value'] > 1.5 else 'compressed'} "
                f"credit risk appetite in the market."
            )

        if t10y2y:
            curve_desc = (
                "inverted (recession signal)" if t10y2y['latest_value'] < 0
                else "flat" if t10y2y['latest_value'] < 0.5
                else "positively sloped"
            )
            narrative_parts.append(
                f"The 10Y-2Y Treasury spread is {t10y2y['latest_value']}%, "
                f"indicating a {curve_desc} yield curve."
            )

        result["narrative"] = " ".join(narrative_parts)

    except Exception as e:
        result["status"]["narrative_error"] = str(e)

    if failed:
        result["status"]["failed_series"] = failed
        result["status"]["success"] = len(failed) < 3
    else:
        result["status"]["success"] = True

    return result


# Test
macro = get_macro_context()

print("STATUS:", macro["status"])
print("\nSNAPSHOT:")
for k, v in macro["snapshot"].items():
    if v:
        print(f"  {k}: {v['latest_value']} ({v['direction']}, {v['qoq_change']:+.2f})")

print(f"\nNARRATIVE:")
print(macro["narrative"])

STATUS: {'success': True}

SNAPSHOT:
  GDP: 31865.72 (up, +443.19)
  UNRATE: 4.27 (down, -0.06)
  FEDFUNDS: 3.63 (down, -0.01)
  BAA10Y: 1.54 (down, -0.08)
  T10Y2Y: 0.33 (down, -0.13)

NARRATIVE:
GDP stood at $31,865.72bn as of 2026-01-01, expanding by $443.19bn quarter over quarter. The unemployment rate is 4.27%, falling 0.06pp quarter over quarter. The Federal Funds Rate stands at 3.63%, indicating a moderate monetary policy environment. The Moody's BAA corporate bond spread is 1.54%, suggesting moderate credit risk appetite in the market. The 10Y-2Y Treasury spread is 0.33%, indicating a flat yield curve.


In [46]:
# Test LlamaIndex + ChromaDB setup
from llama_index.core import VectorStoreIndex, Document
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import chromadb

# Load embedding model
print("Loading embedding model...")
embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")
print("Embedding model loaded")

# Initialize ChromaDB
chroma_client = chromadb.PersistentClient(path="data/chroma")
print("ChromaDB initialized")

# Test with a small piece of text first
test_text = """
Apple Inc. reported strong quarterly results with revenue of $94.9 billion.
The company's gross margin expanded to 46.3% driven by services growth.
Management expressed confidence in the iPhone upgrade cycle continuing.
CEO Tim Cook highlighted AI integration as a key priority for 2025.
The company faces headwinds from China market softness and FX impacts.
Debt to EBITDA remains low at 0.7x with strong free cash flow generation.
"""

# Create a document
doc = Document(text=test_text, metadata={"source": "test", "ticker": "AAPL"})
print(f"Document created: {len(test_text)} chars")

# Create collection
collection = chroma_client.get_or_create_collection("test_collection")
vector_store = ChromaVectorStore(chroma_collection=collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Build index
print("Building index...")
index = VectorStoreIndex.from_documents(
    [doc],
    storage_context=storage_context,
    embed_model=embed_model
)
print("Index built successfully")

# Test query
query_engine = index.as_query_engine()
response = query_engine.query("What did management say about AI?")
print(f"\nTest query response: {response}")

Loading embedding model...
Embedding model loaded
ChromaDB initialized
Document created: 434 chars
Building index...
Index built successfully


ValueError: 
******
Could not load OpenAI model. If you intended to use OpenAI, please check your OPENAI_API_KEY.
Original error:
No API key found for OpenAI.
Please set either the OPENAI_API_KEY environment variable or openai.api_key prior to initialization.
API keys can be found or created at https://platform.openai.com/account/api-keys

******

In [47]:
# Replace the query engine section with this
from llama_index.core import Settings
from llama_index.llms.groq import Groq
import os
from dotenv import load_dotenv

load_dotenv()

# Tell LlamaIndex to use Groq instead of OpenAI
Settings.llm = Groq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)
Settings.embed_model = embed_model

# Now test query
query_engine = index.as_query_engine()
response = query_engine.query("What did management say about AI?")
print(f"\nTest query response: {response}")


Test query response: Management did not directly comment on AI, but CEO Tim Cook highlighted AI integration as a key priority for 2025.


In [48]:
# Full indexer test with real AAPL filing
import sys
sys.path.insert(0, '..')
import src.data.edgar as edgar

# Load the already downloaded AAPL filing
filing_data = edgar.get_filing("AAPL", "0000320193")

print(f"MDA length: {len(filing_data['mda']):,} chars")
print(f"Risk Factors length: {len(filing_data['risk_factors']):,} chars")

# Combine sections into documents for indexing
documents = []

if filing_data["mda"]:
    documents.append(Document(
        text=filing_data["mda"],
        metadata={"section": "mda", "ticker": "AAPL"}
    ))

if filing_data["risk_factors"]:
    documents.append(Document(
        text=filing_data["risk_factors"],
        metadata={"section": "risk_factors", "ticker": "AAPL"}
    ))

if filing_data["full_text"]:
    documents.append(Document(
        text=filing_data["full_text"],
        metadata={"section": "full_text", "ticker": "AAPL"}
    ))

print(f"\nDocuments created: {len(documents)}")
for doc in documents:
    print(f"  {doc.metadata['section']}: {len(doc.text):,} chars")

MDA length: 17,981 chars
Risk Factors length: 68,021 chars

Documents created: 3
  mda: 17,981 chars
  risk_factors: 68,021 chars
  full_text: 500,000 chars


In [49]:
# Build full index with real filing data
# Use ticker-specific collection so each company has its own index

ticker = "AAPL"

# Get or create ticker-specific ChromaDB collection
collection = chroma_client.get_or_create_collection(f"filing_{ticker.lower()}")
vector_store = ChromaVectorStore(chroma_collection=collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

print(f"Building index for {ticker}...")
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    embed_model=embed_model,
    show_progress=True
)
print("Index built successfully")

# Test with credit-relevant queries
queries = [
    "What does management say about liquidity and cash position?",
    "What are the main risk factors that could affect the company?",
    "What is the company's debt structure and covenant situation?",
    "What did management say about future revenue guidance?"
]

retriever = index.as_retriever(similarity_top_k=3)

for query in queries:
    print(f"\n{'='*50}")
    print(f"Query: {query}")
    nodes = retriever.retrieve(query)
    print(f"Chunks retrieved: {len(nodes)}")
    print(f"Top chunk preview: {nodes[0].text[:300]}")

Building index for AAPL...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/235 [00:00<?, ?it/s]

Index built successfully

Query: What does management say about liquidity and cash position?
Chunks retrieved: 3
Top chunk preview: Item 1A. Risk Factors The following summarizes factors that could have a material adverse effect on the Company s business, reputation, results of operations, financial condition and stock price. The Company may not be able to accurately predict, control or mitigate these risks. Statements in this s

Query: What are the main risk factors that could affect the company?
Chunks retrieved: 3
Top chunk preview: Item 1A. Risk Factors The following summarizes factors that could have a material adverse effect on the Company s business, reputation, results of operations, financial condition and stock price. The Company may not be able to accurately predict, control or mitigate these risks. Statements in this s

Query: What is the company's debt structure and covenant situation?
Chunks retrieved: 3
Top chunk preview: | 2025 Form 10-K | 43 Term Debt The Company has o

In [50]:
def build_filing_index(ticker: str, filing_data: dict) -> dict:
    """
    Builds a ChromaDB vector index from filing sections.
    If index already exists for ticker, loads it instead of rebuilding.

    Input:
        ticker (str) — e.g. "AAPL"
        filing_data (dict) — output from edgar.get_filing()
    Output: dict with keys:
        - index: VectorStoreIndex object
        - retriever: retriever object ready for querying
        - status: dict
    """

    result = {
        "index": None,
        "retriever": None,
        "status": {}
    }

    try:
        # Check if index already exists
        existing_collections = [
            c.name for c in chroma_client.list_collections()
        ]
        collection_name = f"filing_{ticker.lower()}"

        if collection_name in existing_collections:
            print(f"Loading existing index for {ticker}...")
            collection = chroma_client.get_collection(collection_name)
            vector_store = ChromaVectorStore(chroma_collection=collection)
            storage_context = StorageContext.from_defaults(
                vector_store=vector_store
            )
            index = VectorStoreIndex.from_vector_store(
                vector_store,
                embed_model=embed_model
            )
            result["status"]["index_source"] = "loaded_existing"

        else:
            print(f"Building new index for {ticker}...")

            # Build documents from filing sections
            documents = []

            if filing_data.get("mda"):
                documents.append(Document(
                    text=filing_data["mda"],
                    metadata={"section": "mda", "ticker": ticker}
                ))

            if filing_data.get("risk_factors"):
                documents.append(Document(
                    text=filing_data["risk_factors"],
                    metadata={"section": "risk_factors", "ticker": ticker}
                ))

            if filing_data.get("full_text"):
                documents.append(Document(
                    text=filing_data["full_text"],
                    metadata={"section": "full_text", "ticker": ticker}
                ))

            if not documents:
                result["status"]["success"] = False
                result["status"]["error"] = "No filing sections available to index"
                return result

            collection = chroma_client.get_or_create_collection(collection_name)
            vector_store = ChromaVectorStore(chroma_collection=collection)
            storage_context = StorageContext.from_defaults(
                vector_store=vector_store
            )

            index = VectorStoreIndex.from_documents(
                documents,
                storage_context=storage_context,
                embed_model=embed_model,
                show_progress=False
            )

            result["status"]["index_source"] = "built_new"
            result["status"]["documents_indexed"] = len(documents)

        # Build retriever
        result["index"]     = index
        result["retriever"] = index.as_retriever(similarity_top_k=3)
        result["status"]["success"] = True
        print(f"Index ready for {ticker}")

    except Exception as e:
        result["status"]["success"] = False
        result["status"]["error"] = str(e)

    return result


def query_filing(retriever, query: str) -> str:
    """
    Queries the filing index and returns relevant text chunks.

    Input:
        retriever — retriever object from build_filing_index()
        query (str) — natural language question
    Output: str — concatenated relevant chunks
    """
    try:
        nodes = retriever.retrieve(query)
        chunks = [node.text for node in nodes]
        return "\n\n---\n\n".join(chunks)
    except Exception as e:
        return f"Query failed: {str(e)}"


# Test full pipeline
# First run will load existing index since we already built it
index_data = build_filing_index("AAPL", filing_data)

print(f"\nStatus: {index_data['status']}")

# Test queries
test_queries = [
    "What does management say about liquidity?",
    "What are the key debt covenants?",
    "What risks does management highlight around competition?"
]

for q in test_queries:
    print(f"\nQuery: {q}")
    result_text = query_filing(index_data["retriever"], q)
    print(f"Result preview: {result_text[:300]}")

Loading existing index for AAPL...
Index ready for AAPL

Status: {'index_source': 'loaded_existing', 'success': True}

Query: What does management say about liquidity?
Result preview: Regulatory changes and other actions that materially adversely affect the Company s business may be announced with little or no advance notice and the Company may not be able to effectively mitigate all adverse impacts from such measures. For example, the Company is subject to changing regulations r

Query: What are the key debt covenants?
Result preview: If we consolidate or merge with or into any other person or sell, transfer, lease or convey all or substantially all of our properties and assets in accordance with the Indentures, the Successor will be substituted for us in the Indentures, with the same effect as if it had been an original party to

Query: What risks does management highlight around competition?
Result preview: Changes to the Company s business in response to the DMA or other laws and r